# IWI experiment · Landsat, nighttime light and wavelet bands of the peak-aligned MODIS NDVI year → IWI with the ten models of 11

This notebook is 11 with the MODIS NDVI year aligned on its peak before the wavelet transform. In 11 the 6 wavelet bands come from the 12 calendar months of the survey year, so the same kind of season gives other values when it comes at another time of the year: NDVI peaks in August–September in the Sahel and in February–April in southern Africa, East Africa has two rainy seasons, and where the season runs across New Year the calendar year cuts it in two. Here the 23 MOD13Q1 composites of every pixel (`MODIS_NDVI.tif` of 01) are first shifted circularly so that the composite with the largest NDVI, the peak of season (POS), sits at index 11, the middle of the 23: the 11 composites before it (176 days) hold the green-up and the start of season, the 11 after it the senescence and the end of season, and the composites pushed past one end of the year come back at the other end. The 6 wavelet bands are then computed from this aligned series as in 11 (single-level `dmey` DWT along time; LFC and HFC minimum, maximum and mean), so they describe the season relative to its peak rather than to the calendar date. The ten models, the clusters, the spatial folds of 03, the training and the outputs are those of 11, so every model is compared with itself in 11, paired by test fold (section 6).

| key | model | start | how the 14 bands go in | training |
|---|---|---|---|---|
| `resnet18` | ResNet-18 (torchvision) | ImageNet | the first convolution takes all 14 bands: RED, GREEN and BLUE get their ImageNet weights, every other band their mean, all × 3/14 (Yeh et al. 2020, as in 05 and 07) | `adam` |
| `resnet34` | ResNet-34 (torchvision) | ImageNet | as `resnet18` | `adam` |
| `resnet50` | ResNet-50 (torchvision) | ImageNet | as `resnet18` | `adam` |
| `swin` | Swin Transformer v2-B (torchgeo `swin_v2_b`) | `Swin_V2_B_Weights.LANDSAT_SI_SATLAS` (SatlasPretrain, Landsat 8/9 Level-1, B1–B11) | Landsat as the 11 Level-1 bands of the weights (see below) through the pretrained patch embedding; TEMP1, NL and the NDVI bands through a new projection that starts at zero; the image is resized to 256 × 256 | `adamw` |
| `unet` | U-Net with an EfficientNet-B3 encoder (torchgeo `unet`) | `Unet_Weights.SENTINEL2_2CLASS_FTW` (Fields of The World: Sentinel-2 B4, B3, B2, B8 at two dates) | RED, GREEN, BLUE and NIR get the weights of their Sentinel-2 band (both dates added), the other 10 bands new weights that start at zero; the decoder gives a 224 × 224 map, and its mean is the IWI | `adam` |
| `vit` | ViT-S/16 (torchgeo `vit_small_patch16_224`) | `ViTSmall16_Weights.LANDSAT_ETM_SR_MOCO` (SSL4EO-L) | as in 08: the 6 SR bands through the pretrained patch embedding, TEMP1, NL and the NDVI bands through a new projection that starts at zero | `adamw` |
| `satclip` | ResNet-18 + SatCLIP location embedding (torchgeo `satclip`) | ImageNet + `SatCLIP_Weights.SATCLIP_VIT16_L40` (fixed) | ResNet-18 as `resnet18`; the 256 SatCLIP values of the cluster's longitude and latitude join its 512 features before the output | `adam` |
| `satclip_only` | SatCLIP location embedding + MLP | `SatCLIP_Weights.SATCLIP_VIT16_L40` (fixed) | no image, only the coordinates: how much of the IWI the location alone explains (the same model as in 11 and 12) | `adam` |
| `mosaiks` | MOSAIKS: random convolutional features (torchgeo `RCF`, empirical) + ridge | no weights: the filters are 2048 whitened 4 × 4 patches of training images | 4096 features of the 14 bands (every band standardised first), ridge regression | ridge, alpha on the validation fold (as in 09) |
| `olmoearth` | OlmoEarth v1 Base (torchgeo `olmoearth_v1`) | `OlmoEarthV1_Weights.BASE` | Landsat as its `landsat` modality (the Level-1 bands as for Swin), one time step, tokens of 8 × 8 pixels (240 m); TEMP1, NL and the NDVI bands through a new projection added to the tokens, starting at zero; IWI from the mean of the output tokens | `adamw` |
| `vgg16bn` | VGG-16 with batch normalisation (torchvision `vgg16_bn`), **baseline** | ImageNet | as `resnet18`: the first convolution takes all 14 bands: RED, GREEN and BLUE get their ImageNet weights, every other band their mean, all × 3/14; global average pooling after the last convolution block gives 512 features | `adam` |
| `mlp` | MLP on band statistics, **baseline** without convolutions | no weights | the mean, standard deviation and 10th, 50th and 90th percentile of every one of the 14 bands over the patch (70 values), standardised (BatchNorm), two hidden layers of 256 (ReLU) | `adam` |

**Baselines**. `vgg16bn` and `mlp` are trained on exactly the same inputs, clusters, folds and recipe as the ten models, from ImageNet weights and from scratch (`TRAIN_FROM_SCRATCH=1`; the MLP has no pretrained weights, so its two runs differ only by training noise). VGG-16 has batch normalisation, so that `adam` with learning rate 1e-3 trains it from either start (the plain VGG-16 of 04 stalled on one fold). The MLP sees only per-band statistics of the patch, not where things are, so its gap to the CNNs measures what spatial patterns add. Their runs go to `RUN_ROOT/vgg16bn/` and `RUN_ROOT/mlp/`, where 16–18 find them.

The table is for the default input, 14 bands. With `PK_INPUT = ndvi23` or `ndvi23_wavelet6` (below) the aligned NDVI values are further bands that go in like the wavelet bands (31 or 37 bands).

**The alignment** (`PK_ALIGN` in the environment, section 3). The peak is the composite with the largest NDVI (ties: the first one). `PEAK_WINDOW` > 1 takes instead the largest circular running mean over that many composites, which one noisy composite cannot move; the series itself is never smoothed.

* `pixel` (default): every pixel gets its own shift, `PEAK_INDEX` − its peak. The shift is the timing of its season: the aligned year keeps the shape of the season (the length of the green-up and of the senescence, a second season, the dry-season level) and loses its date.
* `cluster`: all pixels of a cluster get one shift, that of the peak of the patch-mean series. The regional calendar is removed, and the timing differences within the patch are kept (an irrigated field that peaks in the dry season next to rainfed crops).
* `none`: the 23 composites in calendar order through the same pipeline: the control for the alignment. Against 11 it differs only in the series (the 23 composites instead of 12 monthly means).

A year with two seasons of about the same height has two candidate peaks, and neighbouring pixels may then be aligned on different seasons; section 3b shows how often the pixels of a patch agree with the peak of the patch.

**What the alignment can change.** The 6 wavelet bands are summaries over time: the minimum, maximum and mean of the 42 coefficients of each band. With 23 values and the 62-tap `dmey` filter, most coefficients come from the symmetric extension of the series, so the LFC bands follow the mean and the range of the year and hardly depend on when the season comes; HFC minimum and maximum also depend on the ends of the series (where the calendar year cuts a season) and on the decimation of the DWT. In a synthetic test (seasons of seven shapes at 200 random times of the year, noise 0.02 NDVI) the alignment changed the LFC bands by about 5 % of their spread (the standard deviation of the change divided by that of the band) and HFC minimum and maximum by about a third. Section 3b measures this on the data. The input set (`PK_INPUT` in the environment) can therefore also give the models the aligned year itself:

* `wavelet6` (default): 7 Landsat + NL + the 6 wavelet bands of the aligned series, 14 bands, the layout of 11: against 11 only the wavelet bands change.
* `ndvi23`: 7 Landsat + NL + the 23 aligned NDVI values, 31 bands. Band `NDVI_pos-11` … `NDVI_pos+00` … `NDVI_pos+11` is the NDVI k composites before (−) or after (+) the peak, so the start and the end of the season lie in fixed bands (Table 2's `base_ndvi` row with the aligned year). With `PK_ALIGN = none` these are `NDVI_01` … `NDVI_23`, calendar order.
* `ndvi23_wavelet6`: both, 37 bands.

**The NDVI cache.** 01 downloads MODIS with nearest-neighbour resampling, so a 250 m pixel repeats over about 8 × 8 pixels of the 30 m grid and a patch has only about 750 to 1000 distinct NDVI series. `table1_cache/<sensor>_ndvi23_<alignment>_dmey_symmetric/` keeps, for every distinct series, its 23 aligned values, their 6 wavelet bands and the calendar composite of its peak, and for every cluster the index of the series of each of its 224 × 224 pixels: about 0.2 MB per cluster, shared by the three input sets. The Landsat bands come from the band cache of 04 to 12 and NL from the NL cache, as in 11. In GPU memory the 8 Landsat and NL bands take 32 GB for 40 000 clusters and the NDVI bands a few GB, whatever the input set; a batch looks up the series of every pixel.

**SatCLIP.** torchgeo's SatCLIP is the location encoder of SatCLIP: it turns a longitude and latitude into 256 numbers. It was trained to match Sentinel-2 images of the place, but it does not read an image, so it cannot take the input bands. `satclip` adds it to ResNet-18 on the input bands, and `satclip_only` uses it alone (it reads no band, so it is the `satclip_only` of 11 again: with the same clusters, folds and seed it should give the R² of 11, up to the nondeterminism of the GPU). The folds of 03 put every country into every fold, so `satclip_only` shows how much of the IWI the region alone explains, and `satclip` against `resnet18` what the location adds to the image. The SatCLIP encoder does not learn: its values are an input, standardised with the training clusters of each model. The DHS displacement of the coordinates (up to 2 km for urban, 5 km for rural and 10 km for 1 % of the rural clusters) is small against the scale that SatCLIP L40 resolves (hundreds of km).

**The Landsat bands of Swin and OlmoEarth.** Both were pretrained on Landsat 8/9 Level-1 data, bands B1–B11, as digital numbers (TOA reflectance = DN × 2e-5 − 0.1). 01 downloads surface reflectance and surface temperature (Collection 2 Level 2) of Landsat 5, 7, 8 and 9. The notebook takes the reflectance as if it were TOA reflectance, DN = (reflectance + 0.1) / 2e-5, and fills the bands that are not in the 01 files: B1 (coastal) = BLUE, B8 (panchromatic) = the mean of GREEN and RED, B9 (cirrus) = 5000 (clear sky, about 0), and B10 and B11 (thermal) from TEMP1 with the Planck constants of Landsat 8 TIRS. Swin then gets the normalisation of SatlasPretrain, clip((DN − 4000) / 16320, 0, 1), and OlmoEarth its own (mean ± 2 sd of its pretraining data, from `olmoearth_pretrain_minimal`). For OlmoEarth the B8 tokens are left out by default (`OLMOEARTH_PAN`), which halves the number of tokens. These are approximations (the atmosphere is not added back, Landsat 5 and 7 are taken as Landsat 8), which fine-tuning has to absorb.

**Input normalisation**, before the mapping of each model: as in 04 to 12, fixed numbers. Landsat with the GEE scale factors (surface reflectance; TEMP1 in kelvin / 300), NL clipped to 0–150 and normalised with `NormFunc` of the survey year, the wavelet bands and the aligned NDVI values as they are. ViT gets the SR bands as in 08 (clipped to 0–0.3 and divided by 0.3), U-Net RED, GREEN, BLUE and NIR divided by 0.3 (FTW: Sentinel-2 DN / 3000), and ResNet the values as they are, as in 05.

**Clusters and folds.** The clusters of 11: by default all clusters of 03 from both periods (DMSP up to 2013, VIIRS from 2014), as in 06 to 12: the NL image is that of the survey year, normalised with `NormFunc` of that year. `NL_SENSORS = ['VIIRS']` gives the clusters of 04 and 05. Clusters with a no-data pixel in any of their three files are left out, so the training, validation and test clusters of every fold are those of 11 and 12, and the comparisons of section 6 are paired. Every model is trained five times: test fold f, validation fold f − 1, training on the other three.

| model | test fold | validation fold | training folds | Keras name |
|---|---|---|---|---|
| `test_E` | E | D | A, B, C | Fold12345 |
| `test_A` | A | E | B, C, D | Fold23451 |
| `test_B` | B | A | C, D, E | Fold34512 |
| `test_C` | C | B | D, E, A | Fold45123 |
| `test_D` | D | C | E, A, B | Fold51234 |

**Training.** MSE loss on IWI, batch 64, 100 epochs, no augmentation, and the weights with the lowest validation loss, as in 05 to 08 and 11. The output layer runs in float32, and its bias starts at the mean IWI of the training clusters. `adam` (the CNNs and SatCLIP): Adam with learning rate 1e-3 for all weights, as in 05 to 07. `adamw` (the transformers): AdamW with 5e-5 for the pretrained weights and 1e-3 for the new ones, weight decay 0.05, 5 epochs of warm-up and cosine decay, stochastic depth 0.1, as in 08. The seeds are those of 11, so with `wavelet6` every network starts from the same weights and sees the same batches as in 11.

**Comparisons** (section 6). For every model, the mean test R² of the same model in 11 (the 6 wavelet bands of the 12 calendar months), in this notebook with the other alignments of the same input set, in 13 (the 5 DFT bands of the 23 composites: the mean and the amplitudes of the first four harmonics, which do not change when the season moves, the other way of making the NDVI bands blind to the date) and in 12 (no NDVI band), when they are done, and the gain: the test R² of this run minus theirs, paired by test fold. The gain over the `none` run (`calendar`) is the **alignment gain**.

**Output,** in `landsat_nl_peak_runs/<periods>_<start>_<alignment>_<input>/<model>/test_<fold>/` (by default `DMSP_VIIRS_pretrained_pixel11_wavelet6/`; `none` gives `calendar`):

* `result.json` (with the alignment and the input set), `test_predictions.csv`, and `predictions.csv`: the IWI predicted for every cluster with complete data, with its `role` (train, val or test) in that model.
* `history.csv` and `best.pt` (the weights) of the networks; `mosaiks.pt` (filters, band scaling and ridge) for MOSAIKS.
* `features.csv` (`SAVE_FEATURES`): the layer before the output for every cluster, columns `pk_<model>_feat_0`, … (11: `<model>_feat_0`, …; 12: `ln_<model>_feat_0`, …), so that the files can be merged with those of 06 to 12 on `cluster_id` (not for MOSAIKS).
* Section 6: `pk_results.csv`, `pk_per_fold.csv`, `oof_predictions.csv` (every cluster once, with the prediction of each model whose test fold it is in), `pk_r2.png` and `pk_scatter.png`.

**Running.** Check `WORK_DIR`, `TORCH_HOME` and `PY_PACKAGES` in Settings. The weights and `olmoearth_pretrain_minimal` are those of 11 (section 4; nothing to do if 11 has run). The band and NL caches of 04 to 12 are used as they are, and a missing one is built first; the NDVI cache of an alignment is built by the first job that needs it (a few minutes with 16 cores), and the other array tasks wait for it. Every network saves its state after each epoch, so a stopped job continues where it stopped. For the alignment gain, run the `none` alignment of the same input set too, and 11, 12 and 13 for their comparisons (with the same `NL_SENSORS` and start).

**From random weights.** `TRAIN_FROM_SCRATCH=1` in the environment starts every network from random weights (runs in `<periods>_scratch_<alignment>_<input>`, compared with the scratch runs of 11 and 12); the transformers' weights then learn with `LR_SCRATCH` (5e-5, as in 08; DeiT's rule for training from scratch gives 6e-5). The SatCLIP values stay those of the pretrained encoder (an input, not a trained part), and MOSAIKS has no pretrained weights.

# Import libraries

In [ ]:
import os
import sys
import json
import math
import time
import uuid
import datetime
import multiprocessing

# Same as in 01-10: the GPU image sets PROJ_LIB to a folder without proj.db, which breaks rasterio's CRS handling.
for _var in ('PROJ_LIB', 'PROJ_DATA'):
    if os.environ.get(_var) and not os.path.exists(os.path.join(os.environ[_var], 'proj.db')):
        os.environ.pop(_var)

import numpy as np
import pandas as pd
import pywt
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:                      # progress bars are optional
    def tqdm(it, **kw):
        return it
# torchgeo, timm, segmentation_models_pytorch and olmoearth_pretrain_minimal are imported by the models that use them

# Settings

In [ ]:
# ---- input / output ------------------------------------------------------------------------
WORK_DIR    = '/path/to/data/'
FOLDS_CSV   = os.path.join(WORK_DIR, 'train_clusters_folds.csv')   # written by 03_Spatial_Folds
BAND_CACHE  = os.path.join(WORK_DIR, 'table1_cache')               # the band caches of 04-12 (Landsat, ~2.5 MB per
                                                                  # cluster) and the NDVI caches of section 3 (~0.2 MB)
NL_CACHE    = os.path.join(WORK_DIR, 'table2_cache')               # NL images, ~0.1 MB per cluster (as 04-12)
TORCH_HOME  = '/path/to/torch_home'   # pretrained weights: hub/checkpoints
PY_PACKAGES = '/path/to/python_packages'   # olmoearth_pretrain_minimal,
                                                                  # installed there with pip --target (section 4)

# ---- data (as in 04-12, so that the caches are shared) --------------------------------------------------
TARGET       = 'iwi'                # International Wealth Index of the cluster (0-100), a column of FOLDS_CSV
NL_SENSORS   = ['DMSP', 'VIIRS']    # the periods of 01: DMSP = surveys up to 2013, VIIRS = from 2014. Both: all
                                    # clusters, as in 06-12; ['VIIRS']: the clusters of 04 and 05
NDVI_SERIES  = 'monthly'            # of the band cache of 04-12 (only its Landsat bands are read here); the NDVI year
                                    # of this notebook is always the 23 composites of MODIS_NDVI.tif (section 3)
WAVELET      = 'dmey'               # discrete Meyer, filter length 62, as in the paper and in 11
WAVELET_MODE = 'symmetric'          # pywt's default signal extension, as in 11
NODATA       = -999999.0            # no-data value of the 01 files; clusters with any no-data pixel are left out

# ---- the peak alignment of the NDVI year (section 3). PK_ALIGN in the environment:
# 'pixel': the 23 composites of every pixel are shifted circularly so that its own peak of season sits at PEAK_INDEX;
# 'cluster': all pixels of a cluster by one shift, that of the peak of its patch-mean series; 'none': calendar order,
# the control (the same pipeline without the alignment)
ALIGN       = os.environ.get('PK_ALIGN', 'pixel').strip()
PEAK_INDEX  = 11           # position of the peak after the shift: the middle of the 23 composites (11 before, 11 after)
PEAK_WINDOW = 1            # 1: the peak is the composite with the largest NDVI; 3, 5, ...: the largest circular running
                           # mean over that many composites (one noisy composite cannot move it); the series itself is
                           # not smoothed
N_COMP      = 23           # MOD13Q1 composites of a year (01: they start on DOY 1, 17, ..., 353)

# ---- the input set. PK_INPUT in the environment: 'wavelet6': 7 Landsat + NL + the 6 wavelet bands of the aligned year,
# the 14-band layout of 11; 'ndvi23': 7 Landsat + NL + the 23 aligned NDVI values (31 bands); 'ndvi23_wavelet6': both
# (37 bands)
INPUT_SET = os.environ.get('PK_INPUT', 'wavelet6').strip()

# ---- normalisation of the inputs (fixed numbers, as in 04-12) ------------------------------------------------
# Landsat: the Collection 2 Level 2 scale factors of the GEE catalog (LANDSAT/LC08/C02/T1_L2; the same for Landsat 5,
# 7 and 9): SR_B* (BLUE ... SWIR2) DN * 0.0000275 - 0.2 = surface reflectance; ST_B10 / ST_B6 (TEMP1)
# DN * 0.00341802 + 149.0 = surface temperature in kelvin, divided by TEMP1_DIVISOR. The wavelet bands and the aligned
# NDVI values are used as they are. Nighttime light: clipped to NL_CLIP, then (NL - Mean) / stdDev with NormFunc of the
# survey year. Each model then maps these values to what its pretrained weights expect (section 4).
TEMP1_DIVISOR = 300.0      # kelvin / 300 as in the paper's Keras notebooks
NL_CLIP = (0, 150)         # as in 26_tfrecords_generation_DMSP_VIIRS (VIIRS max 150, DMSP max 63); None: no clipping


def NormFunc(Year):
    if (Year >= 2014):     # VIIRS
        Mean = 0.1326654328849849
        stdDev = 1.7479698750873318
    else:                  # DMSP
        Mean = 4.152879625271373
        stdDev = 3.4762732210548317

    return Mean, stdDev


# ---- training (section 4) -------------------------------------------------------------------------------------
FOLDS         = ['A', 'B', 'C', 'D', 'E']
EPOCHS        = 100
BATCH_SIZE    = 64
L2_HEAD       = 0.0        # weight penalty on the output layer: none, as in 06-08 (04 and 05: 0.01, as in the paper)
LR            = 1e-3       # 'adam' (ResNets, U-Net, SatCLIP): Adam, one learning rate, no schedule, as in 05-07
LR_PRETRAINED = 5e-5       # 'adamw' (ViT, Swin, OlmoEarth), as in 08: AdamW learning rate of the pretrained weights ...
LR_NEW        = 1e-3       # ... and of the new ones (the output layer and the projection of the extra bands)
LR_SCRATCH    = 5e-5       # LR_PRETRAINED with TRAIN_FROM_SCRATCH (08; DeiT from scratch: 5e-4 x 64 / 512 = 6e-5)
WEIGHT_DECAY  = 0.05       # 'adamw': on the pretrained weight matrices (not biases, norms, embeddings, position biases)
WARMUP_EPOCHS = 5          # 'adamw': linear warm-up of the learning rates, then cosine decay to 0 at EPOCHS
DROP_PATH     = 0.1        # stochastic depth of the transformer blocks (ViT, Swin, OlmoEarth)
SEED          = 3127
AMP           = True       # bfloat16 autocast on the GPU for training; features and predictions in float32
GPU_DATA_GB   = 60         # the 8 Landsat and NL bands go into GPU memory when they need less than this (40 000
                           # clusters: 32 GB), otherwise every batch reads them from the caches (slower). The NDVI bands
                           # are always in GPU memory (40 000 clusters: about 5 GB). Lower it if a model runs out of GPU
                           # memory.
N_WORKERS     = len(os.sched_getaffinity(0))   # processes that build a cache if it is missing

# ---- model details (section 4) ---------------------------------------------------------------------------------
SR_MAX           = 0.3          # ViT (SSL4EO-L): reflectance clipped to 0-SR_MAX and divided by SR_MAX, as in 08
FTW_SCALE        = 0.3          # U-Net (FTW): Sentinel-2 DN / 3000 = reflectance / 0.3
SWIN_SIZE        = 256          # Swin: the image is resized to this, so that every window of 8 x 8 tokens is full
OLMOEARTH_SIZE   = 'BASE'       # OlmoEarthV1_Weights: NANO, TINY, BASE or LARGE
OLMOEARTH_PATCH  = 8            # OlmoEarth tokens of 8 x 8 pixels of 30 m: 28 x 28 tokens per image
OLMOEARTH_PAN    = 'missing'    # Landsat B8 (panchromatic) is not in the 01 files: 'missing' leaves its tokens out;
                                # 'copy' fills it with the mean of GREEN and RED (twice the tokens, slower)
MOSAIKS_FEATURES = 4096         # MOSAIKS of torchgeo: 4096 features (2048 filters), kernel 4, bias -1
MOSAIKS_KERNEL   = 4
MOSAIKS_BIAS     = -1.0
MOSAIKS_BATCH    = 8            # images per batch for the features: the convolution output is 0.4 GB per image
ALPHAS           = np.logspace(-2, 6, 33)   # MOSAIKS ridge: the alpha with the lowest validation MSE, as in 09
MLP_QUANTILES    = (0.1, 0.5, 0.9)   # mlp (baseline): percentiles of every band over the patch, besides its mean
                                     # and standard deviation
MLP_HIDDEN       = (256, 256)        # mlp: hidden layers (ReLU) after the standardised band statistics
SAVE_FEATURES    = True         # features.csv of every network: the layer before the output for every cluster

# ---- section 3b: the clusters whose NDVI years are compared before and after the alignment ---------------------------
DIAG_CLUSTERS = 600        # a random sample of the clusters with complete data (all periods together)
PEAK_TOL      = 2          # a pixel agrees with its patch when its own peak is at most this many composites (16 days
                           # each) from the peak of the patch-mean series

# ---- what to run: models ('all', 'none' = only the caches and the results, or keys of MODELS) and test folds ('all'
# or folds), comma-separated. These can also be set in the environment.
RUN_MODELS = os.environ.get('PK_MODELS', 'all')
RUN_FOLDS  = os.environ.get('PK_FOLDS', 'all')
RUN_FOLDS  = FOLDS if RUN_FOLDS == 'all' else [f.strip() for f in RUN_FOLDS.split(',')]
assert set(RUN_FOLDS) <= set(FOLDS), f'unknown test folds: {RUN_FOLDS}'
assert set(NL_SENSORS) <= {'DMSP', 'VIIRS'}, NL_SENSORS
assert OLMOEARTH_PAN in ('missing', 'copy'), OLMOEARTH_PAN
assert ALIGN in ('pixel', 'cluster', 'none'), f'PK_ALIGN must be pixel, cluster or none, not {ALIGN!r}'
assert INPUT_SET in ('wavelet6', 'ndvi23', 'ndvi23_wavelet6'), \
    f'PK_INPUT must be wavelet6, ndvi23 or ndvi23_wavelet6, not {INPUT_SET!r}'
assert 0 <= PEAK_INDEX < N_COMP and PEAK_WINDOW >= 1 and PEAK_WINDOW % 2 == 1, (PEAK_INDEX, PEAK_WINDOW)
KERAS_NAME = {'E': 'Fold12345', 'A': 'Fold23451', 'B': 'Fold34512', 'C': 'Fold45123', 'D': 'Fold51234'}  # 27_Train_NL

# ---- TRAIN_FROM_SCRATCH=1 in the environment: every network starts from random
# weights, the transformers with LR_SCRATCH; the runs go to <periods>_scratch_<alignment>_<input>. check_start() stops
# the notebook when a run folder holds a run of the other kind.
_scratch = os.environ.get('TRAIN_FROM_SCRATCH', '0').strip()
assert _scratch in ('', '0', '1'), f'TRAIN_FROM_SCRATCH must be 0 or 1, not {_scratch!r}'
TRAIN_FROM_SCRATCH = _scratch == '1'
PRETRAINED = not TRAIN_FROM_SCRATCH
if TRAIN_FROM_SCRATCH:
    LR_PRETRAINED = LR_SCRATCH
START = 'scratch' if TRAIN_FROM_SCRATCH else 'pretrained'


def align_tag(align):
    """The name of an alignment in the folders of the NDVI caches and of the runs: calendar (none), pixel11,
    cluster11 (PEAK_WINDOW 3: pixel11w3)."""
    if align == 'none':
        return 'calendar'
    return f'{align}{PEAK_INDEX}' + (f'w{PEAK_WINDOW}' if PEAK_WINDOW > 1 else '')


# ---- output -------------------------------------------------------------------------------------------------
PERIODS   = '_'.join(NL_SENSORS)
ALIGN_TAG = align_tag(ALIGN)
PEAK_RUNS = os.path.join(WORK_DIR, 'landsat_nl_peak_runs')
RUN_ROOT  = os.path.join(PEAK_RUNS, f'{PERIODS}_{START}_{ALIGN_TAG}_{INPUT_SET}')       # one folder per model
# for the comparisons of section 6, paired by test fold, in this order: the same models in 11 (6 wavelet bands of the
# 12 calendar months), in this notebook with the other alignments of the same input set, in 13 (5 DFT bands of the 23
# composites, blind to the timing of the season) and in 12 (no NDVI band)
COMPARE = {'11': os.path.join(WORK_DIR, 'landsat_nl_wavelet_runs', f'{PERIODS}_{START}')}
COMPARE.update({align_tag(a): os.path.join(PEAK_RUNS, f'{PERIODS}_{START}_{align_tag(a)}_{INPUT_SET}')
                for a in ('none', 'pixel', 'cluster') if a != ALIGN})
COMPARE['13'] = os.path.join(WORK_DIR, 'landsat_nl_dft_runs', f'{PERIODS}_{START}')
COMPARE['12'] = os.path.join(WORK_DIR, 'landsat_nl_runs', f'{PERIODS}_{START}')
# and, as in 11: 06 (NL only) and 07 (landsat_wavelet6) on the same clusters, the ridge of 09 on their features, and 05
# (base_wavelet6 with ResNet-18, VIIRS clusters only)
REF_STARTS    = ['scratch', 'imagenet'] if TRAIN_FROM_SCRATCH else ['imagenet', 'scratch']   # first one found is used
RIDGE_RESULTS = os.path.join(WORK_DIR, 'ridge_runs', 'nl08_landsat09', 'ridge_results.csv')
T2_RESNET18   = os.path.join(WORK_DIR, 'table2_resnet18_runs_fixednorm', 'finetune')

os.environ['TORCH_HOME'] = TORCH_HOME
if os.path.isdir(PY_PACKAGES) and PY_PACKAGES not in sys.path:
    sys.path.append(PY_PACKAGES)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(no GPU: only for code tests)')
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| pywt', pywt.__version__,
      '| cache workers:', N_WORKERS)
print('alignment:', ALIGN, '(calendar order)' if ALIGN == 'none' else
      f'(peak of season at index {PEAK_INDEX} of {N_COMP}' + (f', running mean of {PEAK_WINDOW}' if PEAK_WINDOW > 1
                                                             else '') + ')', '| input set:', INPUT_SET)
print('periods:', NL_SENSORS, '| start:', START, '| runs ->', RUN_ROOT)

# 1. Input bands and models

The input bands, in this order: the 7 Landsat bands, NL, and the NDVI bands of the input set (`PK_INPUT`): the 6 wavelet bands of the aligned year in the order of 11 (and of bands 12–17 of the paper's TF records), LFC minimum, maximum, mean, then HFC minimum, maximum, mean; and/or the 23 aligned NDVI values, `NDVI_pos-11` (11 composites, 176 days, before the peak) … `NDVI_pos+00` (the peak) … `NDVI_pos+11` (with `PK_ALIGN = none`: `NDVI_01` … `NDVI_23`, the composites in calendar order, as in 01). `NDVI_CHANNELS` are the values that the NDVI cache keeps for every distinct series: the 23 values, the 6 wavelet bands and `POS`, the calendar composite of its own peak (0–22), which is not an input. `EXTRA` are the bands that the Landsat weights of ViT, Swin and OlmoEarth do not have; they go through the new projection of those models.

In [ ]:
LANDSAT = ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'TEMP1']
N_NDVI = 12 if NDVI_SERIES == 'monthly' else 23
NDVI = [f'NDVI_{k:02d}' for k in range(1, N_NDVI + 1)]
WAVELET_BANDS = ['LFC_min', 'LFC_max', 'LFC_mean', 'HFC_min', 'HFC_max', 'HFC_mean']
CHANNELS = LANDSAT + NDVI + WAVELET_BANDS         # the bands of the band cache of 04-12, in this order
# the 23 composites after the alignment: k composites before (-) or after (+) the peak; calendar order with 'none'
NDVI23 = ([f'NDVI_{k:02d}' for k in range(1, N_COMP + 1)] if ALIGN == 'none' else
          [f'NDVI_pos{k - PEAK_INDEX:+03d}' for k in range(N_COMP)])
NDVI_CHANNELS = NDVI23 + WAVELET_BANDS + ['POS']   # the values of the NDVI cache (section 3), in this order
NDVI_INPUT = {'wavelet6': WAVELET_BANDS, 'ndvi23': NDVI23, 'ndvi23_wavelet6': NDVI23 + WAVELET_BANDS}[INPUT_SET]
BASE = LANDSAT + ['NL']                            # the first 8 input bands, from the band and NL caches of 04-12
INPUT = BASE + NDVI_INPUT                          # the input bands, in this order (wavelet6: as base_wavelet6 of 04)
BAND = {b: k for k, b in enumerate(INPUT)}         # position of every band in the input
SR_BANDS = LANDSAT[:6]                             # surface reflectance, BLUE ... SWIR2
EXTRA = ['TEMP1', 'NL'] + NDVI_INPUT               # not in the Landsat weights of ViT, Swin and OlmoEarth
EXTRA_INDEX = [BAND[b] for b in EXTRA]
assert len(BAND) == len(INPUT), 'a band appears twice in INPUT'

# key: (model, pretrained weights, training recipe, inputs: 'image' = the input bands, 'location' = SatCLIP values)
MODELS = {
    'resnet18':     ('ResNet-18', 'ImageNet (torchvision)', 'adam', ('image',)),
    'resnet34':     ('ResNet-34', 'ImageNet (torchvision)', 'adam', ('image',)),
    'resnet50':     ('ResNet-50', 'ImageNet (torchvision)', 'adam', ('image',)),
    'swin':         ('Swin Transformer v2-B', 'Swin_V2_B_Weights.LANDSAT_SI_SATLAS', 'adamw', ('image',)),
    'unet':         ('U-Net, EfficientNet-B3 encoder', 'Unet_Weights.SENTINEL2_2CLASS_FTW', 'adam', ('image',)),
    'vit':          ('ViT-S/16', 'ViTSmall16_Weights.LANDSAT_ETM_SR_MOCO', 'adamw', ('image',)),
    'satclip':      ('ResNet-18 + SatCLIP location', 'ImageNet + SatCLIP_Weights.SATCLIP_VIT16_L40', 'adam',
                     ('image', 'location')),
    'satclip_only': ('SatCLIP location + MLP', 'SatCLIP_Weights.SATCLIP_VIT16_L40', 'adam', ('location',)),
    'mosaiks':      ('MOSAIKS (RCF) + ridge', 'none (random convolutional features)', 'ridge', ('image',)),
    'olmoearth':    (f'OlmoEarth v1 {OLMOEARTH_SIZE.title()}', f'OlmoEarthV1_Weights.{OLMOEARTH_SIZE}', 'adamw',
                     ('image',)),
    # baselines: a VGG-16 and an MLP without convolutions, on the same inputs
    'vgg16bn':      ('VGG-16 with batch normalisation', 'ImageNet (torchvision vgg16_bn)', 'adam', ('image',)),
    'mlp':          ('MLP on band statistics', 'none (random weights)', 'adam', ('image',)),
}

if RUN_MODELS == 'all':
    todo = list(MODELS)
elif RUN_MODELS == 'none':
    todo = []
else:
    todo = [k.strip() for k in RUN_MODELS.split(',') if k.strip()]
bad = [k for k in todo if k not in MODELS]
assert not bad, f'unknown models: {bad}'
print(f'{len(INPUT)} input bands ({INPUT_SET}, {ALIGN_TAG}):', ', '.join(INPUT))
print('models to run:', todo, '| test folds:', RUN_FOLDS)


def start_of(key):
    """The pretrained weights a model starts from in this run (TRAIN_FROM_SCRATCH: random, except SatCLIP)."""
    if PRETRAINED or MODELS[key][2] == 'ridge':
        return MODELS[key][1]
    return 'random + SatCLIP_VIT16_L40 (fixed)' if 'location' in MODELS[key][3] else 'random'


pd.DataFrame([(k, m, start_of(k), r, ' + '.join(n)) for k, (m, _, r, n) in MODELS.items()],
             columns=['key', 'model', 'start', 'training', 'input'])

# 2. Clusters

All clusters of `train_clusters_folds.csv` of the periods in `NL_SENSORS`. For every period they are in the order of 04 to 10 (the clusters of that NL sensor, sorted by `cluster_id`), so that the caches are shared, and as in 11 and 12. They keep the fold of 03.

In [ ]:
table = pd.read_csv(FOLDS_CSV)
if 'nl_sensor' not in table:                       # 01 uses DMSP up to 2013 and VIIRS from 2014
    table['nl_sensor'] = np.where(table['year'] >= 2014, 'VIIRS', 'DMSP')
if 'folder' not in table:
    table['folder'] = [os.path.join(WORK_DIR, 'Dataset', str(c).replace('/', '_')) for c in table['cluster_id']]
assert TARGET in table, f'{FOLDS_CSV} has no column {TARGET}'

# the clusters of every period in the order of 04-10, then one table: part = period, part_row = row in its caches
PARTS = {s: table[table['nl_sensor'] == s].sort_values('cluster_id').reset_index(drop=True) for s in NL_SENSORS}
clusters = pd.concat([p.assign(part=s, part_row=np.arange(len(p))) for s, p in PARTS.items()], ignore_index=True)
assert clusters['cluster_id'].is_unique, 'a cluster_id appears twice in FOLDS_CSV'
has_target = clusters[TARGET].notna().to_numpy()

mismatch = (clusters['nl_sensor'] == 'VIIRS') != (clusters['year'] >= 2014)
if mismatch.any():
    print(f'WARNING: {mismatch.sum()} clusters whose sensor does not match NormFunc of their year (NormFunc uses the '
          f'year):\n{clusters.loc[mismatch, ["cluster_id", "year", "nl_sensor"]].head().to_string()}\n')
left = table.loc[~table['nl_sensor'].isin(NL_SENSORS), 'nl_sensor']
print(f'{len(clusters)} clusters', clusters['nl_sensor'].value_counts().to_dict(),
      f'| {has_target.sum()} with {TARGET} | {clusters["country"].nunique()} countries, '
      f'years {clusters["year"].min()}-{clusters["year"].max()}'
      + (f' | left out (other period): {left.value_counts().to_dict()}' if len(left) else ''))
print(f'{TARGET}:', clusters.loc[has_target, TARGET].describe().round(2).to_dict())
if clusters['fold'].isna().any():
    print(f"{clusters['fold'].isna().sum()} clusters have no fold: they get predictions but no model is trained or "
          'tested on them')
print()
print(pd.crosstab(clusters['nl_sensor'], clusters['fold'], margins=True).to_string())

# 3. Caches

The Landsat and NL bands come from the two caches of 04 to 12, one folder per period, both float16, as in 11:

* **The band cache** (`table1_cache/<sensor>_monthly_dmey_symmetric/`, about 2.5 MB per cluster): the 7 Landsat bands with the GEE scale factors, surface reflectance (DN × 0.0000275 − 0.2) and TEMP1 in kelvin / 300 ((DN × 0.00341802 + 149) / 300). Its 12 monthly NDVI values and the 6 wavelet bands of 11 are not read here.
* **The NL cache** (`table2_cache/<sensor>_NL/`, about 0.1 MB per cluster): the `NL.tif` image as it is.

The NDVI bands come from the NDVI cache of this notebook (next cell but one). A cluster with a no-data pixel in any of its three files is left out, as in 04 to 12. A complete cache is used as it is, a missing one is built, and a job that finds a cache being built by another job (of 04 to 14) waits for it. The rest of the fixed normalisation is applied when the bands are loaded (`normalise()`): NL clipped to 0–150 and normalised with `NormFunc` of the survey year (and TEMP1 rescaled if `TEMP1_DIVISOR` is not 300). The table at the end of the NDVI cache cell shows the resulting values of every input band for a sample of clusters.

In [ ]:
PATCH = 224
BAND_DIRS = {s: os.path.join(BAND_CACHE, f'{s}_{NDVI_SERIES}_{WAVELET}_{WAVELET_MODE}') for s in NL_SENSORS}
NL_DIRS   = {s: os.path.join(NL_CACHE, f'{s}_NL') for s in NL_SENSORS}


def month_of_composites(year):
    """Calendar month (1-12) in which each of the 23 MOD13Q1 composites of a year starts (DOY 1, 17, ..., 353)."""
    jan1 = datetime.date(int(year), 1, 1)
    return np.array([(jan1 + datetime.timedelta(days=16 * k)).month for k in range(23)])


def cluster_bands(folder, year):
    """The bands of the band cache for one cluster (float32, CHANNELS order) and its mean NL, or None if a file has
    no-data pixels. TEMP1 is kelvin / 300 here; normalise() applies TEMP1_DIVISOR."""
    with rasterio.open(os.path.join(folder, 'Landsat.tif')) as src:
        ls = src.read().astype(np.float32)
    with rasterio.open(os.path.join(folder, 'MODIS_NDVI.tif')) as src:
        nd = src.read().astype(np.float32)
    with rasterio.open(os.path.join(folder, 'NL.tif')) as src:
        nl = src.read(1).astype(np.float64)
    if (ls == NODATA).any() or (nd == NODATA).any() or (nl == NODATA).any():
        return None
    ls[:6] = ls[:6] * 0.0000275 - 0.2                     # surface reflectance
    ls[6] = (ls[6] * 0.00341802 + 149.0) / 300.0          # kelvin / 300
    if NDVI_SERIES == 'monthly':
        month = month_of_composites(year)
        nd = np.stack([nd[month == m].mean(axis=0) for m in range(1, 13)])
    lfc, hfc = pywt.dwt(nd, WAVELET, mode=WAVELET_MODE, axis=0)    # single-level DWT along time, every pixel
    wav = np.stack([lfc.min(0), lfc.max(0), lfc.mean(0), hfc.min(0), hfc.max(0), hfc.mean(0)])
    return np.concatenate([ls, nd, wav]).astype(np.float32), float(nl.mean())


def cluster_nl(folder, year):
    """The NL band of one cluster (float32, shape (1, 224, 224)) and its mean, or None if it has no-data pixels."""
    with rasterio.open(os.path.join(folder, 'NL.tif')) as src:
        nl = src.read(1).astype(np.float64)
    if (nl == NODATA).any():
        return None
    return nl[None].astype(np.float32), float(nl.mean())


COMPUTE = {'bands': cluster_bands, 'nl': cluster_nl}


def _cache_one(task):
    """Compute one cluster and write it at its place in the .part file. Returns the mean NL and the band sums."""
    i, folder, year, offset, kind, part = task
    try:
        out = COMPUTE[kind](folder, year)
    except Exception as e:
        return i, np.nan, None, f'{type(e).__name__}: {e}'[:200]
    if out is None:
        return i, np.nan, None, 'no-data pixels'
    x, nl_mean = out
    fd = os.open(part, os.O_WRONLY)
    try:
        os.pwrite(fd, x.astype(np.float16).tobytes(), offset)
    finally:
        os.close(fd)
    x64 = x.astype(np.float64)
    return i, nl_mean, np.stack([x64.sum(axis=(1, 2)), (x64 ** 2).sum(axis=(1, 2))]), ''


def build_cache(cache_dir, table, channels, kind):
    """features.npy, band_stats.csv and index.csv of one cache folder for the clusters in table (format of 04-10)."""
    t0 = time.time()
    features = os.path.join(cache_dir, 'features.npy')
    shape = (len(table), len(channels), PATCH, PATCH)
    arr = np.lib.format.open_memmap(features + '.part', mode='w+', dtype=np.float16, shape=shape)
    header, per_cluster = arr.offset, int(np.prod(shape[1:])) * 2
    del arr
    tasks = [(i, r.folder, int(r.year), header + i * per_cluster, kind, features + '.part')
             for i, r in enumerate(table.itertuples())]
    nl_mean = np.full(len(table), np.nan)
    sums = np.zeros((2, len(channels)))
    error = [''] * len(table)
    with multiprocessing.get_context('fork').Pool(N_WORKERS) as pool:
        for i, t, s, err in tqdm(pool.imap_unordered(_cache_one, tasks, chunksize=4), total=len(tasks), desc=kind):
            nl_mean[i], error[i] = t, err
            if s is not None:
                sums += s
    ok = np.isfinite(nl_mean)
    n_pix = ok.sum() * PATCH * PATCH
    mean = sums[0] / n_pix
    std = np.sqrt(np.maximum(sums[1] / n_pix - mean ** 2, 1e-12))
    pd.DataFrame({'band': channels, 'mean': mean, 'std': std}).to_csv(os.path.join(cache_dir, 'band_stats.csv'),
                                                                      index=False)
    os.replace(features + '.part', features)
    index = table[['cluster_id', 'country', 'year', 'lat', 'lon']].copy()
    index['nl_mean'], index['ok'], index['error'] = nl_mean, ok, error
    index.to_csv(os.path.join(cache_dir, 'index.csv'), index=False)    # the cache counts as complete once this exists
    print(f'{kind} cache: {ok.sum()} of {len(ok)} clusters in {time.time() - t0:.0f} s -> {features}')


def cache_is_complete(cache_dir, table):
    index, features = os.path.join(cache_dir, 'index.csv'), os.path.join(cache_dir, 'features.npy')
    if not (os.path.exists(index) and os.path.exists(features)):
        return False
    return pd.read_csv(index)['cluster_id'].tolist() == table['cluster_id'].tolist()


def ensure_cache(cache_dir, table, channels, kind):
    """Build a cache folder unless it is complete. Waits when another job is building it (same lock in 04-14)."""
    os.makedirs(cache_dir, exist_ok=True)
    lock = os.path.join(cache_dir, 'building.lock')
    if cache_is_complete(cache_dir, table):
        print(f'{kind} cache found:', cache_dir)
        return
    try:
        os.mkdir(lock)
    except FileExistsError:
        print(f'another job is building {cache_dir} (if none is running, delete {lock}); waiting ...')
        t_wait = time.time()
        while os.path.exists(lock) and time.time() - t_wait < 4 * 3600:
            time.sleep(30)
        assert cache_is_complete(cache_dir, table), 'the cache is still not complete: start the notebook again'
        return
    try:
        if not cache_is_complete(cache_dir, table):     # another job may have finished it just now
            build_cache(cache_dir, table, channels, kind)
    finally:
        os.rmdir(lock)


# ---- reading the Landsat and NL bands and the fixed normalisation (Settings) -------------------------------------
FROM_BANDS = np.array([BAND[b] for b in LANDSAT])                   # their positions in INPUT ...
FROM_BANDS_CH = np.array([CHANNELS.index(b) for b in LANDSAT])      # ... and in the band cache


def read_base(pos):
    """Cached float16 Landsat and NL bands (BASE, the first 8 bands of INPUT) of the clusters at the rows pos of
    `clusters`, shape (len(pos), 8, 224, 224). read_input() (section 3, NDVI cache) adds the NDVI bands."""
    pos = np.asarray(pos)
    out = np.empty((len(pos), len(BASE), PATCH, PATCH), np.float16)
    part, prow = clusters['part'].to_numpy()[pos], clusters['part_row'].to_numpy()[pos]
    for s in np.unique(part):
        k = np.flatnonzero(part == s)
        k = k[np.argsort(prow[k])]                      # read in file order
        out[k[:, None], FROM_BANDS[None, :]] = FEAT[s][prow[k][:, None], FROM_BANDS_CH[None, :]]
        out[k, BAND['NL']] = NLF[s][prow[k], 0]
    return out


def nl_norm_params(pos):
    """NormFunc mean and standard deviation (float32 arrays) of the clusters at the rows pos of `clusters`."""
    params = [NormFunc(int(y)) for y in clusters['year'].to_numpy()[pos]]
    return np.array([m for m, _ in params], np.float32), np.array([s for _, s in params], np.float32)


TEMP_FROM_CACHE = 300.0 / TEMP1_DIVISOR    # the band cache keeps TEMP1 in kelvin / 300


def normalise(x, nl_mean, nl_std):
    """Training values of the input bands, in place (float32 tensor (n, bands, 224, 224), INPUT order, at least the 8
    of BASE): Landsat with the GEE scale factors as cached (TEMP1 in kelvin / TEMP1_DIVISOR), NL clipped to NL_CLIP and
    normalised with NormFunc (nl_mean, nl_std: one value per cluster, from nl_norm_params), the NDVI bands as they
    are."""
    if TEMP_FROM_CACHE != 1.0:
        x[:, BAND['TEMP1']] *= TEMP_FROM_CACHE
    nl = x[:, BAND['NL']] if NL_CLIP is None else x[:, BAND['NL']].clamp(*NL_CLIP)
    x[:, BAND['NL']] = (nl - nl_mean.view(-1, 1, 1)) / nl_std.view(-1, 1, 1)
    return x


for s in NL_SENSORS:
    ensure_cache(BAND_DIRS[s], PARTS[s], CHANNELS, 'bands')
    ensure_cache(NL_DIRS[s], PARTS[s], ['NL'], 'nl')
BAND_INDEX = {s: pd.read_csv(os.path.join(BAND_DIRS[s], 'index.csv')) for s in NL_SENSORS}
NL_INDEX = {s: pd.read_csv(os.path.join(NL_DIRS[s], 'index.csv')) for s in NL_SENSORS}
FEAT = {s: np.load(os.path.join(BAND_DIRS[s], 'features.npy'), mmap_mode='r') for s in NL_SENSORS}
NLF = {s: np.load(os.path.join(NL_DIRS[s], 'features.npy'), mmap_mode='r') for s in NL_SENSORS}
for s in NL_SENSORS:
    assert FEAT[s].shape == (len(PARTS[s]), len(CHANNELS), PATCH, PATCH), (s, FEAT[s].shape)
    assert NLF[s].shape == (len(PARTS[s]), 1, PATCH, PATCH), (s, NLF[s].shape)
    print(f'{s} period: band cache {FEAT[s].shape}, {FEAT[s].nbytes / 1e9:.1f} GB; NL cache {NLF[s].shape}, '
          f'{NLF[s].nbytes / 1e9:.1f} GB')
complete = np.concatenate([BAND_INDEX[s]['ok'].to_numpy(bool) & NL_INDEX[s]['ok'].to_numpy(bool) for s in NL_SENSORS])
errors = pd.Series(np.concatenate([BAND_INDEX[s]['error'].fillna('').to_numpy(str) for s in NL_SENSORS]))
print(f'clusters with complete Landsat, MODIS and NL files: {complete.sum()} of {len(complete)}; left out:',
      errors[~complete].str.split(':').str[0].value_counts().to_dict())
assert complete.any(), 'no cluster has complete data'

## The NDVI cache: the 23 composites aligned on their peak

For every period, `table1_cache/<sensor>_ndvi23_<alignment>_dmey_symmetric/` (`NDVI_DIRS`) holds the NDVI bands of all clusters of that period, in the order of its band cache. `cluster_ndvi()` reads the 23 composites of a cluster's `MODIS_NDVI.tif` and finds its distinct series (01 downloads MODIS with nearest-neighbour resampling, so a 250 m pixel repeats over about 8 × 8 pixels of the 30 m grid). For every distinct series:

1. `find_peak()`: the calendar composite (0–22) of its peak of season, the composite with the largest NDVI (`PEAK_WINDOW` > 1: of the circular running mean; ties: the first);
2. the shift: `PEAK_INDEX` − its own peak (`pixel`), `PEAK_INDEX` − the peak of the patch-mean series, the mean over the 224 × 224 pixels (`cluster`), or 0 (`none`);
3. `roll_series()`: the 23 values shifted circularly, `aligned[k] = ndvi[(k − shift) mod 23]`, which is `np.roll` with one shift per series: the peak goes to `PEAK_INDEX` (with `cluster`, the peak of the patch), and the composites that leave one end of the year come back at the other end;
4. `wavelet_bands()`: the 6 wavelet bands of the aligned series as in 04 to 12, `pywt.dwt(…, 'dmey', mode='symmetric')` along time, then the minimum, maximum and mean of LFC and HFC.

`values.npy` (float32) holds, for every distinct series, the values of `NDVI_CHANNELS`: the 23 aligned values, the 6 wavelet bands and `POS`, the calendar composite of its own peak. `pixels.npy` (uint16) holds, for every cluster, the index of the series of each of its pixels. That is about 0.2 MB per cluster where full maps would take 6 MB, and the cache does not depend on the input set. `index.csv` gives, for every cluster, the first row of its series in `values.npy` (`start`), their number (`n_series`), the calendar composite of the peak of its patch-mean series (`peak_patch`), and whether it has values (`ok`, `error`); `band_stats.csv` the mean and standard deviation of every value over the pixels, and `settings.json` the settings behind the name of the folder.

A complete cache is used as it is; a missing one is built with `N_WORKERS` processes, and a job that finds a cache being built by another job waits for it. A cluster with a no-data pixel in `MODIS_NDVI.tif` is left out (the band cache leaves it out as well). The table at the end shows the training values of every input band for a sample of clusters.

In [ ]:
NDVI_DIRS = {s: os.path.join(BAND_CACHE, f'{s}_ndvi23_{ALIGN_TAG}_{WAVELET}_{WAVELET_MODE}') for s in NL_SENSORS}
NDVI_SETTINGS = {'align': ALIGN, 'peak_index': PEAK_INDEX, 'peak_window': PEAK_WINDOW, 'composites': N_COMP,
                 'wavelet': WAVELET, 'wavelet_mode': WAVELET_MODE, 'channels': NDVI_CHANNELS}


def find_peak(nd):
    """Calendar composite (0 ... 22) of the peak of season of every series of nd (composites first: (23, ...)): the
    composite with the largest NDVI, or with PEAK_WINDOW > 1 the largest circular running mean over PEAK_WINDOW
    composites. Ties: the first."""
    if PEAK_WINDOW > 1:
        h = PEAK_WINDOW // 2
        nd = sum(np.roll(nd, j, axis=0) for j in range(-h, h + 1)) / PEAK_WINDOW
    return np.argmax(nd, axis=0)


def roll_series(nd, shift):
    """np.roll(nd, shift, axis=0) with one shift per series: out[k] = nd[(k - shift) mod 23]. nd: (23, ...); shift: one
    integer, or one per series (the shape of nd[0])."""
    k = np.arange(nd.shape[0]).reshape((-1,) + (1,) * (nd.ndim - 1))
    return np.take_along_axis(nd, np.broadcast_to((k - np.asarray(shift)) % nd.shape[0], nd.shape), axis=0)


def wavelet_bands(nd):
    """The 6 wavelet bands (WAVELET_BANDS order) of the series nd (time first), as in 04-12: single-level DWT along
    time, then the minimum, maximum and mean of LFC and HFC."""
    lfc, hfc = pywt.dwt(nd, WAVELET, mode=WAVELET_MODE, axis=0)
    return np.stack([lfc.min(0), lfc.max(0), lfc.mean(0), hfc.min(0), hfc.max(0), hfc.mean(0)])


def peak_shift(peak, peak_patch, align=ALIGN):
    """The shift of every series (roll_series) for an alignment: from its own peak (pixel) or from the peak of its
    patch-mean series (cluster) to PEAK_INDEX; none: 0. peak: the own peak of every series; peak_patch: one value, or
    one per series."""
    peak = np.asarray(peak)
    if align == 'pixel':
        return PEAK_INDEX - peak
    if align == 'cluster':
        return PEAK_INDEX - np.broadcast_to(np.asarray(peak_patch), peak.shape)
    return np.zeros(peak.shape, int)


def cluster_ndvi(folder):
    """The NDVI cache values (float32 (n, len(NDVI_CHANNELS))) of the n distinct NDVI series of one cluster, the index
    of the series of every pixel (uint16 (224, 224)) and the peak of the patch-mean series, or None if MODIS_NDVI.tif
    has no-data pixels (the band cache leaves such clusters out as well)."""
    with rasterio.open(os.path.join(folder, 'MODIS_NDVI.tif')) as src:
        nd = src.read().astype(np.float32)
    if (nd == NODATA).any():
        return None
    if nd.shape != (N_COMP, PATCH, PATCH):
        raise ValueError(f'MODIS {nd.shape}: expected {N_COMP} composites of {PATCH} x {PATCH} pixels')
    px = np.ascontiguousarray(nd.reshape(N_COMP, -1).T)                               # (pixels, 23)
    key = px.view(np.dtype((np.void, px.dtype.itemsize * N_COMP))).ravel()            # one key per pixel
    _, first, inverse = np.unique(key, return_index=True, return_inverse=True)        # 01: a 250 m pixel repeats ~8 x 8
    inverse = inverse.reshape(-1)
    if len(first) >= 2 ** 15:
        raise ValueError(f'{len(first)} distinct NDVI series, more than the pixel index can hold')
    series = px[first].T.astype(np.float64)                                           # (23, distinct series)
    peak = find_peak(series)                                                          # the own peak of every series
    peak_patch = int(find_peak(nd.reshape(N_COMP, -1).mean(axis=1, dtype=np.float64)))   # of the patch-mean series
    aligned = roll_series(series, peak_shift(peak, peak_patch))
    values = np.concatenate([aligned, wavelet_bands(aligned), peak[None]]).T          # (distinct series, channels)
    return values.astype(np.float32), inverse.reshape(PATCH, PATCH).astype(np.uint16), peak_patch


def _ndvi_one(task):
    """Compute one cluster and write its pixel index at its place in the .part file. Returns its values, the sums over
    its pixels (for band_stats.csv) and the peak of its patch-mean series."""
    i, folder, offset, part = task
    try:
        out = cluster_ndvi(folder)
    except Exception as e:
        return i, None, None, -1, f'{type(e).__name__}: {e}'[:200]
    if out is None:
        return i, None, None, -1, 'no-data pixels'
    v, pix, peak_patch = out
    fd = os.open(part, os.O_WRONLY)
    try:
        os.pwrite(fd, pix.tobytes(), offset)
    finally:
        os.close(fd)
    n_pix = np.bincount(pix.reshape(-1), minlength=len(v)).astype(np.float64)       # pixels of every series
    v64 = v.astype(np.float64)
    return i, v, np.stack([n_pix @ v64, n_pix @ v64 ** 2]), peak_patch, ''


def build_ndvi_cache(cache_dir, table):
    """values.npy, pixels.npy, index.csv, band_stats.csv and settings.json of the NDVI cache of the clusters in table
    (one period, in the order of its band cache)."""
    t0 = time.time()
    pixels, values = os.path.join(cache_dir, 'pixels.npy'), os.path.join(cache_dir, 'values.npy')
    arr = np.lib.format.open_memmap(pixels + '.part', mode='w+', dtype=np.uint16, shape=(len(table), PATCH, PATCH))
    header = arr.offset
    del arr
    tasks = [(i, r.folder, header + i * PATCH * PATCH * 2, pixels + '.part') for i, r in enumerate(table.itertuples())]
    start, count = np.full(len(table), -1, np.int64), np.zeros(len(table), np.int64)
    peak_patch, error = np.full(len(table), -1, np.int64), [''] * len(table)
    sums = np.zeros((2, len(NDVI_CHANNELS)))
    raw, n_rows = values + '.f32.part', 0
    with open(raw, 'wb') as fh, multiprocessing.get_context('fork').Pool(N_WORKERS) as pool:
        for i, v, s, p, err in tqdm(pool.imap(_ndvi_one, tasks, chunksize=4), total=len(tasks), desc='ndvi'):
            error[i] = err
            if v is not None:                       # in cluster order: the same cache for any N_WORKERS
                fh.write(np.ascontiguousarray(v, '<f4').tobytes())
                start[i], count[i], peak_patch[i] = n_rows, len(v), p
                n_rows += len(v)
                sums += s
    with open(values + '.part', 'wb') as fh:        # values.npy: the raw blocks with a header
        np.lib.format.write_array_header_1_0(fh, {'descr': '<f4', 'fortran_order': False,
                                                  'shape': (n_rows, len(NDVI_CHANNELS))})
        with open(raw, 'rb') as src:
            while True:
                b = src.read(1 << 26)
                if not b:
                    break
                fh.write(b)
    os.remove(raw)
    ok = count > 0
    n_pix = max(ok.sum() * PATCH * PATCH, 1)
    mean = sums[0] / n_pix
    std = np.sqrt(np.maximum(sums[1] / n_pix - mean ** 2, 0))
    pd.DataFrame({'band': NDVI_CHANNELS, 'mean': mean, 'std': std}).to_csv(os.path.join(cache_dir, 'band_stats.csv'),
                                                                           index=False)
    with open(os.path.join(cache_dir, 'settings.json'), 'w') as fh:
        json.dump(NDVI_SETTINGS, fh, indent=1)
    os.replace(values + '.part', values)
    os.replace(pixels + '.part', pixels)
    index = table[['cluster_id', 'country', 'year', 'lat', 'lon']].copy()
    index['start'], index['n_series'], index['peak_patch'] = start, count, peak_patch
    index['ok'], index['error'] = ok, error
    index.to_csv(os.path.join(cache_dir, 'index.csv'), index=False)    # the cache counts as complete once this exists
    print(f'NDVI cache: {ok.sum()} of {len(ok)} clusters, {n_rows} distinct series, in {time.time() - t0:.0f} s '
          f'-> {cache_dir}')


def ndvi_cache_is_complete(cache_dir, table):
    files = [os.path.join(cache_dir, f) for f in ('index.csv', 'values.npy', 'pixels.npy', 'settings.json')]
    if not all(os.path.exists(f) for f in files):
        return False
    with open(files[3]) as fh:
        found = json.load(fh)
    if found != json.loads(json.dumps(NDVI_SETTINGS)):
        raise RuntimeError(f'{cache_dir} was made with other settings ({found}): delete it or give it another name')
    return pd.read_csv(files[0])['cluster_id'].tolist() == table['cluster_id'].tolist()


def ensure_ndvi_cache(cache_dir, table):
    """Build an NDVI cache folder unless it is complete. Waits when another job is building it."""
    os.makedirs(cache_dir, exist_ok=True)
    lock = os.path.join(cache_dir, 'building.lock')
    if ndvi_cache_is_complete(cache_dir, table):
        print('NDVI cache found:', cache_dir)
        return
    try:
        os.mkdir(lock)
    except FileExistsError:
        print(f'another job is building {cache_dir} (if none is running, delete {lock}); waiting ...')
        t_wait = time.time()
        while os.path.exists(lock) and time.time() - t_wait < 4 * 3600:
            time.sleep(30)
        assert ndvi_cache_is_complete(cache_dir, table), 'the cache is still not complete: start the notebook again'
        return
    try:
        if not ndvi_cache_is_complete(cache_dir, table):     # another job may have finished it just now
            build_ndvi_cache(cache_dir, table)
    finally:
        os.rmdir(lock)


def read_pixels(pos):
    """The pixel index (uint16 (len(pos), 224, 224)) of the clusters at the rows pos of `clusters`."""
    pos = np.asarray(pos)
    out = np.empty((len(pos), PATCH, PATCH), np.uint16)
    part, prow = clusters['part'].to_numpy()[pos], clusters['part_row'].to_numpy()[pos]
    for s in np.unique(part):
        k = np.flatnonzero(part == s)
        k = k[np.argsort(prow[k])]                      # read in file order
        out[k] = NDVI_PIXELS[s][prow[k]]
    return out


def ndvi_block(i, cols):
    """The NDVI cache values (float32 (n_series, len(cols)); cols: positions in NDVI_CHANNELS) of the distinct series
    of the cluster at the row i of `clusters`."""
    s, prow = clusters.at[i, 'part'], clusters.at[i, 'part_row']
    st, n = NDVI_START[s][prow], NDVI_COUNT[s][prow]
    assert n > 0, f"{clusters.at[i, 'cluster_id']} has no NDVI values"
    return np.asarray(NDVI_VALUES[s][st:st + n][:, cols], np.float32)


def read_ndvi(pos, bands):
    """NDVI bands (names in NDVI_CHANNELS) of the clusters at the rows pos of `clusters` as 224 x 224 maps, float32
    (len(pos), len(bands), 224, 224)."""
    cols = [NDVI_CHANNELS.index(b) for b in bands]
    pos = np.asarray(pos)
    out = np.empty((len(pos), len(cols), PATCH, PATCH), np.float32)
    for k, (i, pix) in enumerate(zip(pos, read_pixels(pos))):
        out[k] = np.moveaxis(ndvi_block(i, cols)[pix], -1, 0)
    return out


def read_input(pos):
    """All input bands (INPUT order) of the clusters at the rows pos of `clusters`, before normalise(): float16
    (len(pos), len(INPUT), 224, 224), the Landsat and NL bands from their caches and the NDVI bands from the NDVI
    cache."""
    return np.concatenate([read_base(pos), read_ndvi(pos, NDVI_INPUT).astype(np.float16)], axis=1)


NDVI_INDEX, NDVI_VALUES, NDVI_PIXELS, NDVI_START, NDVI_COUNT = {}, {}, {}, {}, {}
for s in NL_SENSORS:
    ensure_ndvi_cache(NDVI_DIRS[s], PARTS[s])
    NDVI_INDEX[s] = pd.read_csv(os.path.join(NDVI_DIRS[s], 'index.csv'))
    NDVI_START[s] = NDVI_INDEX[s]['start'].to_numpy(np.int64)
    NDVI_COUNT[s] = NDVI_INDEX[s]['n_series'].to_numpy(np.int64)
    NDVI_VALUES[s] = (np.load(os.path.join(NDVI_DIRS[s], 'values.npy'), mmap_mode='r') if NDVI_COUNT[s].sum()
                      else np.zeros((0, len(NDVI_CHANNELS)), np.float32))
    NDVI_PIXELS[s] = np.load(os.path.join(NDVI_DIRS[s], 'pixels.npy'), mmap_mode='r')
    assert NDVI_VALUES[s].shape == (NDVI_COUNT[s].sum(), len(NDVI_CHANNELS)), (s, NDVI_VALUES[s].shape)
    assert NDVI_PIXELS[s].shape == (len(PARTS[s]), PATCH, PATCH), (s, NDVI_PIXELS[s].shape)
    n_ok = int(NDVI_INDEX[s]['ok'].sum())
    per = f" ({NDVI_INDEX[s].loc[NDVI_INDEX[s]['ok'], 'n_series'].median():.0f} per cluster, median)" if n_ok else ''
    print(f'{s} period: NDVI cache of {n_ok} clusters, {len(NDVI_VALUES[s])} distinct series{per}, '
          f'{(NDVI_VALUES[s].nbytes + NDVI_PIXELS[s].nbytes) / 1e9:.2f} GB -> {NDVI_DIRS[s]}')
ndvi_ok = np.concatenate([NDVI_INDEX[s]['ok'].to_numpy(bool) for s in NL_SENSORS])
ndvi_error = pd.Series(np.concatenate([NDVI_INDEX[s]['error'].fillna('').to_numpy(str) for s in NL_SENSORS]))
lost = complete & ~ndvi_ok
if lost.any():
    print(f'{lost.sum()} clusters with complete files have no NDVI values:',
          ndvi_error[lost].str.split(':').str[0].value_counts().to_dict())
complete = complete & ndvi_ok
print(f'clusters with complete data: {complete.sum()} of {len(complete)}')
assert complete.any(), 'no cluster has complete data'

# training values of every input band for a sample of clusters of every period
rng = np.random.default_rng(0)
sample = np.sort(np.concatenate([rng.choice(c, size=min(16, len(c)), replace=False) for c in
                                 (np.flatnonzero(complete & (clusters['part'] == s).to_numpy()) for s in NL_SENSORS)
                                 if len(c)]))
m, sd = nl_norm_params(sample)
x = normalise(torch.from_numpy(read_input(sample).astype(np.float32)), torch.from_numpy(m), torch.from_numpy(sd))
print(f'\ntraining values of the input bands, {len(sample)} clusters:',
      clusters['part'].to_numpy()[sample].tolist().count('DMSP'), 'DMSP and',
      clusters['part'].to_numpy()[sample].tolist().count('VIIRS'), 'VIIRS period')
pd.DataFrame({'min': x.amin(dim=(0, 2, 3)).numpy(), 'mean': x.mean(dim=(0, 2, 3)).numpy(),
              'max': x.amax(dim=(0, 2, 3)).numpy()}, index=INPUT).T.round(3)

In [ ]:
# check the wealthiest cluster of every period against a fresh computation from its files (the NDVI bands of 64 of its
# pixels also one by one, with np.roll), then show its bands and the NDVI year of some of its pixels before and after
# the alignment
DOY_START = 1 + 16 * np.arange(N_COMP)                 # the first day of every composite (01)
LINE_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']


def circular_distance(a, b, n=N_COMP):
    """The distance between composites a and b on the circular year (0 ... 11)."""
    d = np.abs(np.asarray(a) - np.asarray(b)) % n
    return np.minimum(d, n - d)


def pick_series(pix, pos, centre, n=4):
    """Up to n distinct series of a patch for the plot: the one at the centre, then the largest ones whose peaks lie at
    least 2 composites from those already picked (then the largest ones left)."""
    count = np.bincount(pix.reshape(-1), minlength=len(pos))
    order = np.argsort(-count, kind='stable')
    picked = [centre]
    for j in order:
        if len(picked) < n and j not in picked and all(circular_distance(pos[j], pos[q]) >= 2 for q in picked):
            picked.append(j)
    for j in order:
        if len(picked) < n and j not in picked:
            picked.append(j)
    return picked


usable = complete & has_target
fig, ax = plt.subplots(len(NL_SENSORS), 6, figsize=(24, 3.9 * len(NL_SENSORS)), constrained_layout=True, squeeze=False)
for r, s in enumerate(NL_SENSORS):
    cand = np.flatnonzero(usable & (clusters['part'] == s).to_numpy())
    if not len(cand):
        for a in ax[r]:
            a.axis('off')
        continue
    i = int(cand[np.argmax(clusters[TARGET].to_numpy()[cand])])
    folder, prow = clusters.at[i, 'folder'], clusters.at[i, 'part_row']
    x_new, nl_mean_new = cluster_bands(folder, clusters.at[i, 'year'])
    nl_new, _ = cluster_nl(folder, clusters.at[i, 'year'])
    x = read_base([i])[0].astype(np.float32)
    assert np.allclose(x[FROM_BANDS], x_new[FROM_BANDS_CH], rtol=2e-3, atol=2e-3)
    assert np.allclose(x[BAND['NL']], nl_new[0], rtol=2e-3, atol=2e-3)
    assert np.isclose(nl_mean_new, BAND_INDEX[s].at[prow, 'nl_mean'])
    # the NDVI cache against cluster_ndvi() ...
    v_new, pix_new, peak_patch = cluster_ndvi(folder)
    v_cache = ndvi_block(i, list(range(len(NDVI_CHANNELS))))
    assert (NDVI_PIXELS[s][prow] == pix_new).all() and np.allclose(v_cache, v_new, rtol=1e-6, atol=1e-7)
    assert NDVI_INDEX[s].at[prow, 'peak_patch'] == peak_patch
    # ... and against np.roll, pixel by pixel, from the file
    with rasterio.open(os.path.join(folder, 'MODIS_NDVI.tif')) as src:
        nd = src.read().astype(np.float64)
    assert peak_patch == find_peak(nd.reshape(N_COMP, -1).mean(axis=1))
    maps = read_ndvi([i], NDVI_CHANNELS)[0]
    for y_, x_ in zip(*np.random.default_rng(r).integers(0, PATCH, (2, 64))):
        v = nd[:, y_, x_]
        own = int(find_peak(v))
        aligned = np.roll(v, {'pixel': PEAK_INDEX - own, 'cluster': PEAK_INDEX - peak_patch, 'none': 0}[ALIGN])
        lfc, hfc = pywt.dwt(aligned, WAVELET, mode=WAVELET_MODE)
        assert np.allclose(maps[:N_COMP, y_, x_], aligned, rtol=1e-6, atol=1e-7)
        assert np.allclose(maps[N_COMP:N_COMP + 6, y_, x_], [lfc.min(), lfc.max(), lfc.mean(), hfc.min(), hfc.max(),
                                                             hfc.mean()], rtol=1e-5, atol=1e-6)
        assert maps[NDVI_CHANNELS.index('POS'), y_, x_] == own
        if ALIGN == 'pixel' and PEAK_WINDOW == 1:
            assert aligned[PEAK_INDEX] == aligned.max()                   # the peak sits at PEAK_INDEX
    print(f"{s} period: caches match the files for {clusters.at[i, 'cluster_id']} ({len(v_new)} distinct NDVI series; "
          f'64 pixels computed one by one agree)')

    # the plots: Landsat, NL, the peak of every pixel, the NDVI year of four series before and after the alignment
    pos = v_cache[:, NDVI_CHANNELS.index('POS')].astype(int)
    pick = pick_series(pix_new, pos, int(pix_new[PATCH // 2, PATCH // 2]))
    stored = v_cache[:, :N_COMP]                                       # as in this run's cache
    calendar = roll_series(stored.T, -peak_shift(pos, peak_patch)).T  # back to calendar order
    shown = stored if ALIGN != 'none' else roll_series(calendar.T, peak_shift(pos, peak_patch, 'pixel')).T
    weight = np.bincount(pix_new.reshape(-1), minlength=len(pos)) / pix_new.size
    ax[r, 0].imshow(np.moveaxis(np.clip(x[[BAND['RED'], BAND['GREEN'], BAND['BLUE']]] / 0.3, 0, 1), 0, -1))
    ax[r, 0].set_title(f"{clusters.at[i, 'cluster_id']} ({clusters.at[i, 'year']}), {TARGET} "
                       f"{clusters.at[i, TARGET]:.1f}\nLandsat RGB", fontsize=9)
    fig.colorbar(ax[r, 1].imshow(x[BAND['NL']], cmap='magma'), ax=ax[r, 1], shrink=0.8)
    ax[r, 1].set_title(f'{s} nighttime light, as downloaded', fontsize=9)
    cb = fig.colorbar(ax[r, 2].imshow(maps[NDVI_CHANNELS.index('POS')], cmap='twilight', vmin=-0.5,
                                      vmax=N_COMP - 0.5, interpolation='nearest'), ax=ax[r, 2], shrink=0.8)
    cb.set_ticks([0, 90 / 16, 181 / 16, 273 / 16])                    # composites that start on these days
    cb.set_ticklabels(['1 Jan', '1 Apr', '1 Jul', '1 Oct'])
    ax[r, 2].set_title(f'peak of season of every pixel (POS)\npatch-mean series: composite {peak_patch} '
                       f'(day {DOY_START[peak_patch]})', fontsize=9)
    for c, (series, xs, xlabel, title) in enumerate((
            (calendar, DOY_START, 'first day of the composite', 'NDVI year in calendar order (01)'),
            (shown, (np.arange(N_COMP) - PEAK_INDEX) * 16, 'days from the peak',
             {'pixel': 'aligned on the own peak (this run)', 'cluster': 'aligned on the peak of the patch (this run)',
              'none': 'aligned on the own peak (not used: PK_ALIGN = none)'}[ALIGN]))):
        a = ax[r, 3 + c]
        for color, j in zip(LINE_COLORS, pick):
            a.plot(xs, series[j], '-o', color=color, linewidth=2, markersize=3,
                   label=('centre, ' if j == pick[0] else '') + f'peak {DOY_START[pos[j]]}')
        a.plot(xs, weight @ series, '--', color='#0b0b0b', linewidth=1.5, label='patch mean')
        a.set_xlabel(xlabel, fontsize=8)
        a.set_ylabel('NDVI', fontsize=8)
        a.set_title(title, fontsize=9)
        a.tick_params(labelsize=7)
        a.grid(color='#e1e0d9', linewidth=0.8)
        a.set_axisbelow(True)
        for spine in ('top', 'right'):
            a.spines[spine].set_visible(False)
    ax[r, 4].axvline(0, color='#52514e', linewidth=1, linestyle=':')
    ax[r, 3].legend(frameon=False, fontsize=7, title='series (day of its peak)', title_fontsize=7)
    fig.colorbar(ax[r, 5].imshow(maps[NDVI_CHANNELS.index('HFC_max')], cmap='viridis'), ax=ax[r, 5], shrink=0.8)
    ax[r, 5].set_title(f'HFC_max of this run ({ALIGN_TAG})', fontsize=9)
    for c in (0, 1, 2, 5):
        ax[r, c].set_xticks([])
        ax[r, c].set_yticks([])
fig.suptitle('The wealthiest cluster of every period: bands as cached, and the NDVI year before and after the '
             'alignment')
plt.show()

# 3b. What the alignment does

Three views of the NDVI years before any training. The first uses all clusters with complete data, the others a random sample of `DIAG_CLUSTERS` of them, every pixel counted once (a distinct series weighs as many pixels as it covers).

1. **When the seasons peak.** The calendar composite of the peak of every cluster's patch-mean NDVI year, per country (countries from north to south), and, in the sample, the share of a patch's pixels whose own peak lies at most `PEAK_TOL` composites from it. Clusters of one country that peak at different times of the year give the calendar-order wavelet bands of 11 different parts of their seasons. Where the share is low, the pixels of a patch peak at different times (two seasons, irrigated or evergreen land beside rainfed crops), and the `pixel` and `cluster` alignments differ.
2. **The mean NDVI year** of the sample in four latitude bands, in calendar order and aligned on the own peak of every pixel.
3. **The wavelet bands** of every pixel of the sample, computed from its NDVI year in calendar order (`calendar`), with each alignment, and from the 12 calendar months of 11 (`11`):
   * the correlation of every band of this run (`ALIGN_TAG`) with the same band of the others: a band that the alignment hardly changes cannot carry a gain in section 6;
   * the share of every band that is the calendar date of the season: the R² of the band on the cosine and sine of the composite of the pixel's own peak (the date of the season also differs between regions, so this is not zero even where the band is blind to it);
   * the share of every band that the level and the range of the NDVI year explain: the R² of the band on the mean, minimum, maximum and standard deviation of the 23 composites, which do not depend on the order of the composites. The rest is what the wavelet band adds to them.

The tables go to `pk_alignment_bands.csv` in `RUN_ROOT`.

In [ ]:
# a random sample of the clusters with complete data: their distinct NDVI series, with the number of pixels of each
rng = np.random.default_rng(SEED)
diag = np.sort(rng.choice(np.flatnonzero(complete), size=min(DIAG_CLUSTERS, int(complete.sum())), replace=False))
D = {k: [] for k in ('row', 'weight', 'pos', 'peak_patch', 'year', 'stored', 'wavelet')}
for i, pix in zip(diag, read_pixels(diag)):
    v = ndvi_block(i, list(range(len(NDVI_CHANNELS))))
    D['row'].append(np.full(len(v), i))
    D['weight'].append(np.bincount(pix.reshape(-1), minlength=len(v)).astype(np.float64))
    D['pos'].append(v[:, NDVI_CHANNELS.index('POS')].astype(int))
    D['peak_patch'].append(np.full(len(v), NDVI_INDEX[clusters.at[i, 'part']].at[clusters.at[i, 'part_row'],
                                                                                 'peak_patch']))
    D['year'].append(np.full(len(v), int(clusters.at[i, 'year'])))
    D['stored'].append(v[:, :N_COMP].astype(np.float64))
    D['wavelet'].append(v[:, [NDVI_CHANNELS.index(b) for b in WAVELET_BANDS]].astype(np.float64))
D = {k: np.concatenate(v) for k, v in D.items()}
w = D['weight']

# every series in calendar order, then aligned in each way: the year as the models of each run see it
calendar = roll_series(D['stored'].T, -peak_shift(D['pos'], D['peak_patch'])).T
YEARS = {align_tag(a): roll_series(calendar.T, peak_shift(D['pos'], D['peak_patch'], a)).T
         for a in ('none', 'pixel', 'cluster')}
WAV = {t: wavelet_bands(y.T).T for t, y in YEARS.items()}
assert np.allclose(WAV[ALIGN_TAG], D['wavelet'], rtol=1e-5, atol=1e-6), 'the cache does not match its own alignment'
monthly = np.empty((len(calendar), 12))                       # the 12 calendar months of 11 (04: cluster_bands())
for y in np.unique(D['year']):
    m, sel = month_of_composites(y), D['year'] == y
    monthly[sel] = np.stack([calendar[sel][:, m == k].mean(axis=1) for k in range(1, 13)], axis=1)
WAV['11'] = wavelet_bands(monthly.T).T


def wls_r2(y, X, w):
    """Weighted R2 of y on the columns of X (and an intercept)."""
    X1 = np.c_[np.ones(len(y)), X] * np.sqrt(w)[:, None]
    beta = np.linalg.lstsq(X1, y * np.sqrt(w), rcond=None)[0]
    res = y - np.c_[np.ones(len(y)), X] @ beta
    return 1 - np.average(res ** 2, weights=w) / np.average((y - np.average(y, weights=w)) ** 2, weights=w)


def wcorr(a, b, w):
    a, b = a - np.average(a, weights=w), b - np.average(b, weights=w)
    return np.average(a * b, weights=w) / np.sqrt(np.average(a * a, weights=w) * np.average(b * b, weights=w))


angle = 2 * np.pi * D['pos'] / N_COMP
TIMING = np.c_[np.cos(angle), np.sin(angle)]                  # the calendar date of every pixel's own peak
LEVEL = np.c_[calendar.mean(1), calendar.min(1), calendar.max(1), calendar.std(1)]   # the same in every order
others = [t for t in WAV if t != ALIGN_TAG]
tables = {'correlation with this run': pd.DataFrame({t: [wcorr(WAV[ALIGN_TAG][:, b], WAV[t][:, b], w)
                                                         for b in range(6)] for t in others}, index=WAVELET_BANDS),
          'R2 on the date of the peak': pd.DataFrame({t: [wls_r2(WAV[t][:, b], TIMING, w) for b in range(6)]
                                                      for t in WAV}, index=WAVELET_BANDS),
          'R2 on the NDVI mean, minimum, maximum and sd': pd.DataFrame({t: [wls_r2(WAV[t][:, b], LEVEL, w)
                                                                            for b in range(6)] for t in WAV},
                                                                       index=WAVELET_BANDS)}
os.makedirs(RUN_ROOT, exist_ok=True)
out = os.path.join(RUN_ROOT, 'pk_alignment_bands.csv')           # every array task writes it: a unique temporary name
pd.concat(tables, axis=1).rename_axis('band').reset_index().to_csv(tmp := f'{out}.{uuid.uuid4().hex[:8]}.tmp',
                                                                   index=False, float_format='%.5g')
os.replace(tmp, out)
agree = pd.Series(np.bincount(np.searchsorted(diag, D['row']),
                              weights=w * (circular_distance(D['pos'], D['peak_patch']) <= PEAK_TOL))
                  / np.bincount(np.searchsorted(diag, D['row']), weights=w), index=diag)
print(f'{len(diag)} clusters, {len(w)} distinct NDVI series, {int(w.sum())} pixels. Share of the pixels of a patch '
      f'whose own peak is at most {PEAK_TOL} composites from the peak of the patch-mean year: median '
      f'{agree.median():.2f}, quartiles {agree.quantile(0.25):.2f}-{agree.quantile(0.75):.2f}')
LEGEND = {'calendar': 'the 23 composites in calendar order',
          align_tag('pixel'): 'aligned on the own peak of every pixel',
          align_tag('cluster'): 'aligned on the peak of the patch-mean year', '11': 'the 12 calendar months of 11'}
print('the wavelet bands of ' + '; '.join(f'{t}: {LEGEND[t]}' for t in WAV) + f'. This run: {ALIGN_TAG}.')
for name, t in tables.items():
    print(f'\n{name}')
    display(t.round(3))

In [ ]:
# 1. when the seasons peak (all clusters with complete data), and how many pixels of a patch peak with it (the sample)
info = pd.concat([NDVI_INDEX[s] for s in NL_SENSORS], ignore_index=True)   # in the order of `clusters`
assert (info['cluster_id'].to_numpy() == clusters['cluster_id'].to_numpy()).all()
info = info[complete]
north_to_south = info.groupby('country')['lat'].mean().sort_values(ascending=False).index
share = (pd.crosstab(info['country'], info['peak_patch'], normalize='index')
         .reindex(index=north_to_south, columns=range(N_COMP), fill_value=0.0))
n_country = info['country'].value_counts().reindex(north_to_south)
agree_country = agree.groupby(clusters.loc[agree.index, 'country'].to_numpy()).agg(['median', 'size'])
agree_country = agree_country.reindex(north_to_south)
agree_country.loc[agree_country['size'].fillna(0) < 3, 'median'] = np.nan     # too few clusters in the sample
BLUES = LinearSegmentedColormap.from_list(
    'blues', ['#fcfcfb', '#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b'])
MONTH_TICKS = ([0, 90 / 16, 181 / 16, 273 / 16], ['1 Jan', '1 Apr', '1 Jul', '1 Oct'])   # composites starting then

fig, (a, b) = plt.subplots(1, 2, figsize=(14, 0.24 * len(share) + 1.8), sharey=True, constrained_layout=True,
                           gridspec_kw={'width_ratios': [4, 1]})
im = a.imshow(share.to_numpy(), aspect='auto', cmap=BLUES, vmin=0, vmax=max(float(share.to_numpy().max()), 1e-9),
              interpolation='nearest')
fig.colorbar(im, ax=a, location='bottom', shrink=0.35, aspect=30, pad=0.02,
             label="share of the country's clusters whose patch-mean year peaks in the composite")
a.set_yticks(range(len(share)))
a.set_yticklabels([f'{c} ({n})' for c, n in zip(share.index, n_country)], fontsize=8)
a.set_xticks(*MONTH_TICKS)
a.set_xlabel('composite of the peak of the patch-mean NDVI year (01: 23 composites from 1 January)', fontsize=9)
a.set_title('When the seasons peak: clusters per country (north to south) and composite of the peak', loc='left',
            fontsize=10)
yy = np.arange(len(share))
b.barh(yy, agree_country['median'].to_numpy(), height=0.6, color='#2a78d6')
b.set_xlim(0, 1)
b.set_xlabel(f'pixels within {PEAK_TOL} composites\nof the patch peak (median)', fontsize=9)
b.set_title(f'sample of {len(diag)} clusters', loc='left', fontsize=10)
b.grid(axis='x', color='#e1e0d9', linewidth=0.8)
b.set_axisbelow(True)
for spine in ('top', 'right'):
    b.spines[spine].set_visible(False)
plt.show()

# 2. the mean NDVI year in four latitude bands, in calendar order and aligned on the own peak of every pixel
lat = clusters['lat'].to_numpy()[D['row']]
LAT_BANDS = [('north of 10° N', lat > 10), ('0-10° N', (lat > 0) & (lat <= 10)), ('0-10° S', (lat <= 0) & (lat > -10)),
             ('south of 10° S', lat <= -10)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True, constrained_layout=True)
for a, (year, xs, xlabel, title) in zip(axes, (
        (calendar, DOY_START, 'first day of the composite', 'calendar order (as 01, and 11 by month)'),
        (YEARS[align_tag('pixel')], (np.arange(N_COMP) - PEAK_INDEX) * 16, 'days from the own peak of the pixel',
         'aligned on the own peak of every pixel (pixel)'))):
    for color, (name, sel) in zip(LINE_COLORS, LAT_BANDS):
        if (sel & (w > 0)).any():
            n = len(np.unique(D['row'][sel]))
            a.plot(xs, np.average(year[sel], axis=0, weights=w[sel]), '-o', color=color, linewidth=2, markersize=3,
                   label=f'{name} ({n} clusters)')
    a.set_xlabel(xlabel, fontsize=9)
    a.set_title(title, loc='left', fontsize=10)
    a.grid(color='#e1e0d9', linewidth=0.8)
    a.set_axisbelow(True)
    for spine in ('top', 'right'):
        a.spines[spine].set_visible(False)
axes[0].set_ylabel('mean NDVI of the pixels', fontsize=9)
axes[1].axvline(0, color='#52514e', linewidth=1, linestyle=':')
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle(f'The mean NDVI year of the sample ({len(diag)} clusters)', fontsize=11)
plt.show()

# 4. Models

Every network has the same interface: `features(x, loc)` gives the values of the layer before the output for a batch (`x`: the normalised input bands, `loc`: the SatCLIP values), and `head`, a linear layer that runs in float32, turns them into the IWI. The model-specific mapping of the input bands (table at the top) happens inside the model, so all models read the same batches.

* **ResNet-18/34/50** (`ResNetRegressor`): torchvision's ResNet with a first convolution for the input bands, as the `finetune` mode of 05 and 07; global average pooling gives 512 (ResNet-50: 2048) features. `satclip` adds the 256 standardised SatCLIP values.
* **Swin v2-B** (`SwinRegressor`): torchgeo's `swin_v2_b` with the SatlasPretrain Landsat weights (single image). The patch embedding becomes an `ExtraProjection`: the pretrained 4 × 4 convolution on the 11 Level-1 bands made by `landsat_l1()`, plus a new 4 × 4 convolution of the `EXTRA` bands that starts at zero, so the model starts as the pretrained one. torchvision's Swin v2-B has stochastic depth up to 0.5; it is scaled to `DROP_PATH`. Features: the 1024 values after the final norm and average pooling.
* **U-Net** (`UNetRegressor`): torchgeo's `unet` (segmentation_models_pytorch) with the FTW weights. Its first convolution gets the input bands (RED, GREEN, BLUE, NIR: the weights of B4, B3, B2, B8 of both dates added, which is the same image at both dates; the other bands start at zero). The segmentation head is replaced by the 16 decoder channels averaged over the image and standardised (BatchNorm without learned scale: the averages of BatchNorm + ReLU maps hardly differ between images, and the output layer would need a long time to reach a useful scale) and `head`. In evaluation mode this is the same as a linear map of the 16 decoder channels to an IWI per pixel, averaged over the image; `wealth_map()` gives that map.
* **ViT-S/16** (`ViTRegressor`): as `LandsatViT` of 08, built with torchgeo's `vit_small_patch16_224`: the class token after the final norm, 384 features.
* **SatCLIP** (`satclip_embeddings()`, `LocationMLP`): torchgeo's `satclip` location encoder turns the longitude and latitude of every cluster into 256 values once (section 5); they are standardised with the mean and standard deviation of the training clusters of each model. `satclip_only` feeds them to two hidden layers of 256 (ReLU).
* **MOSAIKS** (`run_mosaiks()`, next cell but one): every band is standardised with the mean and standard deviation of the training clusters' pixels, torchgeo's `RCF` (empirical, as its `MOSAIKS` class: kernel 4, bias −1) takes 2048 random 4 × 4 patches of all input bands of training images, whitened (ZCA), as filters, and every image gives 4096 features (the mean of the positive and of the negative part of each filter response). A ridge regression (features standardised, alpha from `ALPHAS` with the lowest MSE on the validation fold, then fitted on the training and validation folds, as in 09) predicts the IWI. Filters and scaling come from the training folds only, so the test fold is never seen.
* **OlmoEarth v1** (`OlmoEarthRegressor`): torchgeo's `olmoearth_v1`; only its encoder is kept. The Landsat bands go in as the `landsat` modality, one time step (timestamp July), the bands of `landsat_l1()` normalised as in OlmoEarth's pretraining, with `input_res` = 30 m. `TokensWithExtra` adds a new projection of the `EXTRA` bands (8 × 8 convolution, starting at zero) to the Landsat tokens. Features: the mean of the encoder's output tokens (768 for Base).
* **VGG-16-BN** (`VGGRegressor`, baseline): torchvision's `vgg16_bn` with `VGG16_BN_Weights.IMAGENET1K_V1` (or random weights); its first 3 × 3 convolution takes the input bands like the ResNets' first convolution; global average pooling after the last block gives 512 features; the ImageNet classifier is not used.
* **MLP** (`BandStatsMLP`, baseline): no convolution; for every input band, the mean, standard deviation and `MLP_QUANTILES` percentiles of its pixels, standardised by BatchNorm and fed to the `MLP_HIDDEN` hidden layers (ReLU): 256 features.

A run trains the model of one test fold. After every epoch it writes the history (with the learning rates), the best weights (`best.pt`, lowest validation loss) and its full state (`last.pt`). Then it loads `best.pt`, computes the prediction (and the features) of every cluster in float32 and writes `predictions.csv`, `features.csv`, `test_predictions.csv` and `result.json`, and deletes `last.pt`, which is only needed to continue a stopped run. A run with `result.json` (and `features.csv`, with `SAVE_FEATURES`) is skipped. A run with only a `last.pt` continues from that epoch. A finished run whose `features.csv` was deleted writes it again from `best.pt`.

**Before the first run** (in case the machine has no internet access; nothing to do if 11 has run):

```bash
# the weights, into TORCH_HOME/hub/checkpoints (the next cell lists the files that are missing)
D=/path/to/torch_home/hub/checkpoints
mkdir -p $D
wget -nc -P $D https://download.pytorch.org/models/resnet18-f37072fd.pth \
    https://download.pytorch.org/models/resnet34-b627a593.pth \
    https://download.pytorch.org/models/resnet50-0676ba61.pth \
    https://download.pytorch.org/models/vgg16_bn-6c64b313.pth \
    https://hf.co/torchgeo/satlas/resolve/081d6607431bf36bdb59c223777cbb267131b8f2/landsat_swinb_si-4af978f6.pth \
    https://hf.co/torchgeo/ftw/resolve/2ff807f35ec4e04ab329cd66de6117d8b4c26578/commercial/2-class/sentinel2_unet_effb3-3ce575ed.pth \
    https://hf.co/torchgeo/ssl4eo_landsat/resolve/1c88bb51b6e17a21dde5230738fa38b74bd74f76/vits16_landsat_etm_sr_moco-eaa4674e.pth \
    https://huggingface.co/microsoft/SatCLIP-ViT16-L40/resolve/0ef2acc0b91d3c8cfdf4a0cc03207095989287ab/satclip-vit16-l40.ckpt \
    https://huggingface.co/allenai/OlmoEarth-v1-Base/resolve/4bd1392a4539404d2c74276c39f3cb4cfff466cc/weights-551c1cc53337c6faaddead88071d7ebd2bd53ec271600fa6f0ee0a518c8b6e11.pth

# OlmoEarth needs olmoearth_pretrain_minimal: install it into PY_PACKAGES
pip install --no-deps --target /path/to/python_packages \
    olmoearth-pretrain-minimal==0.1.5 universal-pathlib pathlib-abc
```

In [ ]:
# ---- pretrained weights: torchvision and torchgeo read them from TORCH_HOME/hub/checkpoints ------------------------
CKPT_DIR = os.path.join(TORCH_HOME, 'hub', 'checkpoints')
OLMOEARTH_HINT = ('OlmoEarth needs olmoearth_pretrain_minimal. Install it once:\n  '
                  'pip install --no-deps --target ' + PY_PACKAGES +
                  ' olmoearth-pretrain-minimal==0.1.5 universal-pathlib pathlib-abc')


def tg_weights(enum_name, member):
    """A weights entry of torchgeo (0.10 or later)."""
    import torchgeo.models
    return getattr(getattr(torchgeo.models, enum_name), member)


def weights_of(key):
    """The pretrained weights (objects with .url) a model reads in this run."""
    tv = torchvision.models
    resnet = {18: tv.ResNet18_Weights.IMAGENET1K_V1, 34: tv.ResNet34_Weights.IMAGENET1K_V1,
              50: tv.ResNet50_Weights.IMAGENET1K_V1}
    need = []
    if PRETRAINED:
        if key in ('resnet18', 'resnet34', 'resnet50'):
            need.append(resnet[int(key[6:])])
        elif key == 'satclip':
            need.append(resnet[18])
        elif key == 'swin':
            need.append(tg_weights('Swin_V2_B_Weights', 'LANDSAT_SI_SATLAS'))
        elif key == 'unet':
            need.append(tg_weights('Unet_Weights', 'SENTINEL2_2CLASS_FTW'))
        elif key == 'vit':
            need.append(tg_weights('ViTSmall16_Weights', 'LANDSAT_ETM_SR_MOCO'))
        elif key == 'olmoearth':
            need.append(tg_weights('OlmoEarthV1_Weights', OLMOEARTH_SIZE))
        elif key == 'vgg16bn':
            need.append(tv.VGG16_BN_Weights.IMAGENET1K_V1)
    if 'location' in MODELS[key][3]:          # the SatCLIP values are an input, also with TRAIN_FROM_SCRATCH
        need.append(tg_weights('SatCLIP_Weights', 'SATCLIP_VIT16_L40'))
    return need


def download_hint(key):
    try:
        urls = [w.url for w in weights_of(key)]
    except Exception as e:
        return f'(the weights of {key} could not be listed: {type(e).__name__}: {e})'
    return ('If this machine has no internet access, download the weights first, then run again:\n  mkdir -p '
            + CKPT_DIR + ''.join(f'\n  wget -O {os.path.join(CKPT_DIR, os.path.basename(u))} {u}' for u in urls))


# ---- Landsat for the models pretrained on Landsat 8/9 Level-1 data (Swin, OlmoEarth) ---------------------------
L1_BANDS = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11']
TIRS = ((774.8853, 1321.0789), (480.8883, 1201.1442))   # K1, K2 of Landsat 8 TIRS B10 and B11 (MTL files)
CIRRUS_DN = 5000.0                                     # B9 (cirrus) of a clear sky: TOA reflectance about 0
SATLAS_LANDSAT = (4000.0, 16320.0)                     # SatlasPretrain: clip((DN - 4000) / 16320, 0, 1)


def landsat_l1(x):
    """Approximate Landsat 8/9 Level-1 digital numbers of B1-B11, (n, 11, H, W), from the normalised input bands x
    (n, bands, H, W): surface reflectance taken as TOA reflectance, DN = (reflectance + 0.1) / 2e-5; B1 (coastal) =
    BLUE, B8 (panchromatic) = mean of GREEN and RED, B9 (cirrus) = CIRRUS_DN; B10 and B11 = the radiance of TEMP1
    (Planck, TIRS constants) as DN = (L - 0.1) / 3.342e-4."""
    blue, green, red, nir, swir1, swir2 = ((x[:, :6] + 0.1) / 2e-5).unbind(1)
    kelvin = (x[:, BAND['TEMP1']] * TEMP1_DIVISOR).clamp(min=150.0)
    b10, b11 = [(k1 / torch.expm1(k2 / kelvin) - 0.1) / 3.342e-4 for k1, k2 in TIRS]
    return torch.stack([blue, blue, green, red, nir, swir1, swir2, (green + red) / 2, torch.full_like(blue, CIRRUS_DN),
                        b10, b11], dim=1)


def swin_input(x):
    """The 11 bands of the SatlasPretrain Landsat weights, normalised as in SatlasPretrain."""
    return ((landsat_l1(x) - SATLAS_LANDSAT[0]) / SATLAS_LANDSAT[1]).clamp(0, 1)


def vit_input(x):
    """The six SR bands of the SSL4EO-L weights (ETM+ SR_B1-B5, B7), as in 08: clipped to 0-SR_MAX, / SR_MAX."""
    return (x[:, :6] / SR_MAX).clamp(0, 1)


# ---- the models ---------------------------------------------------------------------------------------------------
class Regressor(nn.Module):
    """Common part of the networks: features(x, loc) -> (n, n_features), and `head`, the linear output in float32."""

    def make_head(self, n_features):
        self.n_features = n_features
        self.head = nn.Linear(n_features, 1)
        nn.init.xavier_uniform_(self.head.weight)              # Keras defaults, as in 05-08; the bias is set to the
        nn.init.zeros_(self.head.bias)                         # mean IWI of the training clusters in run_fold

    def forward(self, x, loc=None):
        f = self.features(x, loc)
        with torch.autocast(device_type=f.device.type, enabled=False):    # output layer in float32
            return self.head(f.float()).squeeze(1)

    @staticmethod
    def is_new(name):
        """Parameters that are not pretrained (adamw: LR_NEW): the output layer, the projection of the EXTRA bands."""
        return name.startswith('head.') or '.extra.' in name

    def set_location_scale(self, loc):
        """Standardise the SatCLIP values with the mean and standard deviation of the training clusters (loc)."""
        self.loc_mean.copy_(loc.mean(dim=0))
        self.loc_std.copy_(loc.std(dim=0).clamp(min=1e-6))


def first_conv_weights(bands, rgb_weights):
    """First-layer weights for any bands from the ImageNet ones (Yeh et al. 2020): RED, GREEN and BLUE keep their
    own, every other band gets their mean, and all are scaled by 3 / number of bands (05, 07)."""
    rgb_index = {'RED': 0, 'GREEN': 1, 'BLUE': 2}            # channel order of the ImageNet weights
    mean = rgb_weights.mean(dim=1)
    w = torch.stack([rgb_weights[:, rgb_index[b]] if b in rgb_index else mean for b in bands], dim=1)
    return w * 3.0 / len(bands)


LOC_DIM = 256                                                  # SatCLIP values per location


class ResNetRegressor(Regressor):
    """torchvision ResNet-18/34/50 whose first convolution takes the input bands (the finetune mode of 05 and 07).
    location=True (satclip): the 256 standardised SatCLIP values join the pooled features before the output."""

    def __init__(self, depth, location=False):
        super().__init__()
        net = getattr(torchvision.models, f'resnet{depth}')(weights='IMAGENET1K_V1' if PRETRAINED else None)
        rgb_weights = net.conv1.weight.detach().clone()        # (64, 3, 7, 7)
        net.conv1 = nn.Conv2d(len(INPUT), 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            if PRETRAINED:
                net.conv1.weight.copy_(first_conv_weights(INPUT, rgb_weights))
            else:                                              # torchvision's initialisation of its convolutions
                nn.init.kaiming_normal_(net.conv1.weight, mode='fan_out', nonlinearity='relu')
        n_features = net.fc.in_features
        net.fc = nn.Identity()                                 # the network now ends with the pooled features
        self.backbone = net
        self.location = location
        if location:
            self.register_buffer('loc_mean', torch.zeros(LOC_DIM))
            self.register_buffer('loc_std', torch.ones(LOC_DIM))
            n_features += LOC_DIM
        self.make_head(n_features)

    def features(self, x, loc=None):
        f = self.backbone(x)
        if self.location:
            f = torch.cat([f, ((loc - self.loc_mean) / self.loc_std).to(f.dtype)], dim=1)
        return f


class LocationMLP(Regressor):
    """satclip_only: the standardised SatCLIP values of the cluster's coordinates, two hidden layers (ReLU)."""

    def __init__(self, hidden=256):
        super().__init__()
        self.register_buffer('loc_mean', torch.zeros(LOC_DIM))
        self.register_buffer('loc_std', torch.ones(LOC_DIM))
        self.mlp = nn.Sequential(nn.Linear(LOC_DIM, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.make_head(hidden)

    def features(self, x, loc):
        return self.mlp((loc - self.loc_mean) / self.loc_std)


class VGGRegressor(Regressor):
    """vgg16bn (baseline): torchvision's VGG-16 with batch normalisation. Its first convolution takes all input bands
    like the ResNets: RED, GREEN and BLUE keep their ImageNet weights, every other band gets their mean, all scaled by
    3 / number of bands. The convolution layers end in global average pooling: 512 features (the 4096-wide ImageNet
    classifier is not used, as the ResNets' fc layer)."""

    def __init__(self):
        super().__init__()
        net = torchvision.models.vgg16_bn(weights='IMAGENET1K_V1' if PRETRAINED else None)
        conv = net.features[0]                                 # Conv2d(3, 64, 3, padding=1)
        first = nn.Conv2d(len(INPUT), conv.out_channels, kernel_size=3, padding=1)
        with torch.no_grad():
            if PRETRAINED:
                first.weight.copy_(first_conv_weights(INPUT, conv.weight.detach()))
                first.bias.copy_(conv.bias.detach())
            else:                                              # torchvision's initialisation of the VGG convolutions
                nn.init.kaiming_normal_(first.weight, mode='fan_out', nonlinearity='relu')
                nn.init.zeros_(first.bias)
        net.features[0] = first
        self.backbone = nn.Sequential(net.features, nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.make_head(512)

    def features(self, x, loc=None):
        return self.backbone(x)


class BandStatsMLP(Regressor):
    """mlp (baseline without convolutions): every input band of the patch is summarised by its mean, its standard
    deviation and its MLP_QUANTILES percentiles over the 224 x 224 pixels. These values are standardised (BatchNorm,
    learned from the training batches) and go through the MLP_HIDDEN hidden layers (ReLU). The MLP sees what the
    bands of a patch contain, but not where: its gap to the CNNs measures what the spatial patterns add."""

    def __init__(self):
        super().__init__()
        n_in = len(INPUT) * (2 + len(MLP_QUANTILES))
        self.norm = nn.BatchNorm1d(n_in)
        layers, width = [], n_in
        for h in MLP_HIDDEN:
            layers += [nn.Linear(width, h), nn.ReLU()]
            width = h
        self.mlp = nn.Sequential(*layers)
        self.make_head(width)

    @staticmethod
    def band_statistics(x):
        """(n, C, H, W) -> (n, C * (2 + len(MLP_QUANTILES))), float32: the mean and standard deviation of every band,
        then its percentiles MLP_QUANTILES (the sorted pixel values at the nearest rank)."""
        v = x.float().flatten(2)                               # (n, C, H * W)
        last = v.shape[2] - 1
        ranks = [min(max(int(round(q * last)), 0), last) for q in MLP_QUANTILES]
        q = v.sort(dim=2).values[:, :, ranks]                  # (n, C, len(MLP_QUANTILES))
        return torch.cat([v.mean(dim=2), v.std(dim=2), q.flatten(1)], dim=1)

    def features(self, x, loc=None):
        x = x
        with torch.autocast(device_type=x.device.type, enabled=False):    # statistics and BatchNorm in float32
            z = self.norm(self.band_statistics(x))
        return self.mlp(z)


class ExtraProjection(nn.Module):
    """A pretrained patch embedding (a convolution) fed by `prepare` (input bands -> the bands of the weights), plus a
    new convolution of the same shape for the EXTRA bands, without bias and starting at zero, so that the model
    starts as the pretrained one (TRAIN_FROM_SCRATCH: both with PyTorch's random initialisation)."""

    def __init__(self, pretrained, prepare):
        super().__init__()
        self.pretrained = pretrained
        self.prepare = prepare
        self.extra = nn.Conv2d(len(EXTRA), pretrained.out_channels, kernel_size=pretrained.kernel_size,
                               stride=pretrained.stride, padding=pretrained.padding, bias=False)
        if PRETRAINED:
            nn.init.zeros_(self.extra.weight)

    def forward(self, x):
        return self.pretrained(self.prepare(x)) + self.extra(x[:, EXTRA_INDEX])


class SwinRegressor(Regressor):
    """torchgeo's Swin v2-B with the SatlasPretrain Landsat weights (single image), 1024 features."""

    def __init__(self):
        super().__init__()
        from torchgeo.models import Swin_V2_B_Weights, swin_v2_b
        from torchvision.ops import StochasticDepth
        net = swin_v2_b(Swin_V2_B_Weights.LANDSAT_SI_SATLAS if PRETRAINED else None)
        if not PRETRAINED:                                     # torchgeo sets the 11-band embedding only with weights
            net.features[0][0] = nn.Conv2d(len(L1_BANDS), net.features[0][0].out_channels, kernel_size=4, stride=4)
        assert net.features[0][0].in_channels == len(L1_BANDS), net.features[0][0]
        net.features[0][0] = ExtraProjection(net.features[0][0], swin_input)
        depth = [m for m in net.modules() if isinstance(m, StochasticDepth)]
        top = max(m.p for m in depth)                          # torchvision: 0 ... 0.5 over the blocks
        for m in depth:
            m.p = m.p / top * DROP_PATH
        n_features = net.head.in_features
        net.head = nn.Identity()                               # features -> norm -> average pooling: 1024 values
        self.backbone = net
        self.make_head(n_features)

    def features(self, x, loc=None):
        if SWIN_SIZE != x.shape[-1]:
            x = F.interpolate(x, size=(SWIN_SIZE, SWIN_SIZE), mode='bilinear', align_corners=False)
        return self.backbone(x)


class UNetRegressor(Regressor):
    """torchgeo's U-Net (segmentation_models_pytorch, EfficientNet-B3 encoder) with the FTW weights. The 16 decoder
    channels are averaged over the image and standardised (`pool_norm`: BatchNorm without learned scale, so that the
    output layer starts on a useful scale; the averages of BatchNorm + ReLU maps hardly differ between images): these
    are the 16 features. In evaluation mode this is the mean over the image of an IWI per pixel (wealth_map())."""

    FTW_BANDS = ['RED', 'GREEN', 'BLUE', 'NIR']                # B4, B3, B2, B8, at each of the two dates

    def __init__(self):
        super().__init__()
        from torchgeo.models import Unet_Weights, unet
        net = unet(Unet_Weights.SENTINEL2_2CLASS_FTW if PRETRAINED else None, encoder_name='efficientnet-b3',
                   in_channels=2 * len(self.FTW_BANDS), classes=2)
        stem = next(m for m in net.encoder.modules() if isinstance(m, nn.Conv2d))
        assert stem.in_channels == 2 * len(self.FTW_BANDS), stem
        w = stem.weight.detach()                               # (40, 8, 3, 3)
        new = torch.zeros(w.shape[0], len(INPUT), *w.shape[2:])
        if PRETRAINED:                                         # the same image at both dates
            for j, b in enumerate(self.FTW_BANDS):
                new[:, BAND[b]] = w[:, j] + w[:, j + len(self.FTW_BANDS)]
        else:
            nn.init.kaiming_normal_(new, mode='fan_out', nonlinearity='relu')
        stem.weight = nn.Parameter(new)
        stem.in_channels = len(INPUT)
        n_features = net.segmentation_head[0].in_channels       # 16
        net.segmentation_head = nn.Identity()                  # net(x) is then the decoder output
        self.net = net
        self.register_buffer('ftw_index', torch.tensor([BAND[b] for b in self.FTW_BANDS]), persistent=False)
        self.pool_norm = nn.BatchNorm1d(n_features, affine=False)
        self.make_head(n_features)

    def decoded(self, x):
        x = x.clone()
        x[:, self.ftw_index] = x[:, self.ftw_index] / FTW_SCALE    # as Sentinel-2 DN / 3000
        return self.net(x)                                     # (n, 16, 224, 224)

    def features(self, x, loc=None):
        return self.pool_norm(self.decoded(x).mean(dim=(2, 3)))

    def wealth_map(self, x):
        """IWI per pixel, (n, 224, 224), in evaluation mode; its mean over the image is the prediction."""
        bn = self.pool_norm
        scale = (bn.running_var + bn.eps).sqrt().view(1, -1, 1, 1)
        z = (self.decoded(x).float() - bn.running_mean.view(1, -1, 1, 1)) / scale
        return torch.einsum('nchw,c->nhw', z, self.head.weight[0].float()) + self.head.bias.float()


class ViTRegressor(Regressor):
    """ViT-S/16 with the SSL4EO-L MoCo weights (LandsatViT of 08): the class token after the final norm, 384 values."""

    def __init__(self):
        super().__init__()
        from torchgeo.models import ViTSmall16_Weights, vit_small_patch16_224
        net = vit_small_patch16_224(ViTSmall16_Weights.LANDSAT_ETM_SR_MOCO if PRETRAINED else None,
                                    in_chans=len(SR_BANDS), num_classes=0, drop_path_rate=DROP_PATH)
        net.patch_embed.proj = ExtraProjection(net.patch_embed.proj, vit_input)
        self.backbone = net
        self.make_head(net.num_features)

    def features(self, x, loc=None):
        return self.backbone(x)


class TokensWithExtra(nn.Module):
    """OlmoEarth's patch embeddings plus a new projection of the EXTRA bands, added to the Landsat tokens of every band
    set; it starts at zero (TRAIN_FROM_SCRATCH: random). OlmoEarthRegressor hands over the bands in `extra_input`."""

    def __init__(self, inner, dim, patch):
        super().__init__()
        self.inner = inner
        self.extra = nn.Conv2d(len(EXTRA), dim, kernel_size=patch, stride=patch, bias=False)
        if PRETRAINED:
            nn.init.zeros_(self.extra.weight)
        self.extra_input = None

    def forward(self, input_data, patch_size):
        out = self.inner.forward(input_data, patch_size)
        if self.extra_input is not None:
            e = self.extra(self.extra_input).permute(0, 2, 3, 1)         # (n, h, w, dim)
            out['landsat'] = out['landsat'] + e[:, :, :, None, None, :].to(out['landsat'].dtype)
        return out


class OlmoEarthRegressor(Regressor):
    """The encoder of OlmoEarth v1 on the Landsat bands (its `landsat` modality, one time step); features: the mean of
    its output tokens (768 values for Base)."""

    def __init__(self):
        super().__init__()
        from torchgeo.models import OlmoEarthV1_Weights, olmoearth_v1
        from olmoearth_pretrain_minimal import Normalizer
        from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.constants import Modality
        from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.datatypes import MaskedOlmoEarthSample, MaskValue
        from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.nn.flexi_vit import PoolingType
        self._sample, self._pooling = MaskedOlmoEarthSample, PoolingType.MEAN
        self._online, self._missing = MaskValue.ONLINE_ENCODER.value, MaskValue.MISSING.value
        full = olmoearth_v1(getattr(OlmoEarthV1_Weights, OLMOEARTH_SIZE) if PRETRAINED else None,
                            model_size=OLMOEARTH_SIZE.lower(), drop_path=DROP_PATH)
        encoder = full.model.encoder                           # the decoder and target of pretraining are not needed
        del full
        order = Modality.LANDSAT.band_order                    # ['B8', 'B1', ..., 'B7', 'B9', 'B10', 'B11']
        stats = Normalizer(std_multiplier=2.0).norm_config['landsat']    # OlmoEarth: mean +- 2 sd -> 0 ... 1
        mean = torch.tensor([stats[b]['mean'] for b in order], dtype=torch.float32)
        std = torch.tensor([stats[b]['std'] for b in order], dtype=torch.float32)
        self.register_buffer('l1_order', torch.tensor([L1_BANDS.index(b) for b in order]), persistent=False)
        self.register_buffer('lo', (mean - 2 * std).view(1, -1, 1, 1), persistent=False)
        self.register_buffer('span', (4 * std).view(1, -1, 1, 1), persistent=False)
        groups = encoder.patch_embeddings.tokenization_config.get_bandset_indices('landsat')
        self.n_sets = len(groups)
        self.pan_sets = [k for k, g in enumerate(groups) if list(g) == [order.index('B8')]]   # band sets of B8 alone
        encoder.patch_embeddings = TokensWithExtra(encoder.patch_embeddings, encoder.embedding_size, OLMOEARTH_PATCH)
        self.encoder = encoder
        self.make_head(encoder.embedding_size)

    def sample(self, x):
        """The OlmoEarth input of a batch: landsat (n, H, W, 1, 11), its mask (n, H, W, 1, band sets), timestamps."""
        n, _, h, w = x.shape
        landsat = ((landsat_l1(x)[:, self.l1_order] - self.lo) / self.span).permute(0, 2, 3, 1).unsqueeze(3)
        mask = torch.full((n, h, w, 1, self.n_sets), self._online, dtype=torch.long, device=x.device)
        if OLMOEARTH_PAN == 'missing':
            mask[..., self.pan_sets] = self._missing
        timestamps = torch.tensor([15, 6, 2018], device=x.device).expand(n, 1, 3)   # day, month (0 = Jan), year
        return self._sample(timestamps=timestamps, landsat=landsat.contiguous(), landsat_mask=mask)

    def features(self, x, loc=None):
        tokens = self.encoder.patch_embeddings
        tokens.extra_input = x[:, EXTRA_INDEX]
        try:
            out = self.encoder(self.sample(x), patch_size=OLMOEARTH_PATCH, input_res=30,
                               fast_pass=OLMOEARTH_PAN == 'copy')    # with masked tokens, the slower path
        finally:
            tokens.extra_input = None
        return out['tokens_and_masks'].pool_unmasked_tokens(self._pooling, spatial_pooling=False)


def build_model(key):
    """A new network for a model key, with its pretrained weights (TRAIN_FROM_SCRATCH: random)."""
    try:
        if key in ('resnet18', 'resnet34', 'resnet50'):
            return ResNetRegressor(int(key[6:]))
        if key == 'satclip':
            return ResNetRegressor(18, location=True)
        if key == 'satclip_only':
            return LocationMLP()
        if key == 'swin':
            return SwinRegressor()
        if key == 'unet':
            return UNetRegressor()
        if key == 'vit':
            return ViTRegressor()
        if key == 'olmoearth':
            return OlmoEarthRegressor()
        if key == 'vgg16bn':
            return VGGRegressor()
        if key == 'mlp':
            return BandStatsMLP()
    except ImportError as e:
        raise RuntimeError(f'{key}: a package is missing ({e}).' + ('\n' + OLMOEARTH_HINT if key == 'olmoearth' else
                                                                    '')) from e
    except Exception as e:
        raise RuntimeError(f'{key} could not be built ({type(e).__name__}: {e}).\n{download_hint(key)}') from e
    raise ValueError(f'{key} is not a network')


def satclip_embeddings(pos):
    """The 256 values of torchgeo's SatCLIP location encoder (SATCLIP_VIT16_L40) for the longitude and latitude of the
    clusters at the rows pos of `clusters` (float32, on DEVICE)."""
    try:
        from torchgeo.models import SatCLIP_Weights, satclip
        encoder = satclip(SatCLIP_Weights.SATCLIP_VIT16_L40)
    except Exception as e:
        raise RuntimeError(f'the SatCLIP encoder could not be built ({type(e).__name__}: {e}).\n'
                           f'{download_hint("satclip_only")}') from e
    encoder = encoder.eval().to(DEVICE)
    lonlat = torch.tensor(clusters[['lon', 'lat']].to_numpy(np.float64)[np.asarray(pos)], dtype=torch.float32)
    with torch.no_grad():
        emb = torch.cat([encoder(lonlat[s:s + 4096].to(DEVICE)).float() for s in range(0, len(lonlat), 4096)])
    assert emb.shape == (len(lonlat), LOC_DIM), emb.shape
    return emb


# the weights files of the models to run
missing = {}
for key in todo:
    try:
        for w in weights_of(key):
            f = os.path.join(CKPT_DIR, os.path.basename(w.url))
            if not os.path.exists(f):
                missing[f] = w.url
    except Exception as e:
        print(f'{key}: the weights could not be listed ({type(e).__name__}: {e})')
if missing:
    print('weights not yet in TORCH_HOME (downloaded when a model needs them if this machine has internet access, otherwise the '
          'model stops). To download them beforehand:\n  mkdir -p ' + CKPT_DIR
          + ''.join(f'\n  wget -O {f} {u}' for f, u in missing.items()))
elif todo:
    print('all weights of the models to run are in', CKPT_DIR)
if 'olmoearth' in todo:
    try:
        import olmoearth_pretrain_minimal
        print('olmoearth_pretrain_minimal', getattr(olmoearth_pretrain_minimal, '__version__', ''), 'from',
              os.path.dirname(olmoearth_pretrain_minimal.__file__))
    except ImportError:
        print('WARNING: olmoearth_pretrain_minimal is not installed. ' + OLMOEARTH_HINT)

In [ ]:
class Inputs:
    """The inputs of the models for the clusters at the rows `rows` of `clusters`: the input bands, normalised as in
    04-12 (normalise()), and `loc`, the SatCLIP values of their coordinates (set in section 5 when a model needs them).
    The 8 Landsat and NL bands go into DEVICE memory when they take less than GPU_DATA_GB (otherwise every batch reads
    them from the caches, which is slower). The NDVI bands are always in DEVICE memory, as the values of the distinct
    series of all clusters (float16) and, for every cluster, the index of the series of each pixel (int16): a batch
    looks up the series of every pixel, so the models get the full 224 x 224 maps."""

    def __init__(self, rows, load_images=True):
        self.rows = np.asarray(rows)
        nl_m, nl_s = nl_norm_params(self.rows)
        self.nl_mean = torch.from_numpy(nl_m).to(DEVICE)       # NormFunc of every cluster, by position in rows
        self.nl_std = torch.from_numpy(nl_s).to(DEVICE)
        self.x, self.loc, self.vals = None, None, None
        if load_images:
            gb = len(self.rows) * len(BASE) * PATCH * PATCH * 2 / 1e9
            if gb < GPU_DATA_GB:
                self.x = torch.empty((len(self.rows), len(BASE), PATCH, PATCH), dtype=torch.float16, device=DEVICE)
                for s in tqdm(range(0, len(self.rows), 256), desc=f'loading Landsat and NL, {gb:.1f} GB',
                              leave=False):
                    self.x[s:s + 256] = torch.from_numpy(read_base(self.rows[s:s + 256])).to(DEVICE)
            # the NDVI bands: the values of the distinct series of the clusters, one block per cluster in the order of
            # rows (start: the first row of every block), and the series of every pixel within its cluster's block
            cols = [NDVI_CHANNELS.index(b) for b in NDVI_INPUT]
            blocks = [ndvi_block(i, cols).astype(np.float16)
                      for i in tqdm(self.rows, desc=f'loading {len(NDVI_INPUT)} NDVI bands', leave=False)]
            n_series = np.array([len(v) for v in blocks])
            self.vals = torch.from_numpy(np.concatenate(blocks)).to(DEVICE)
            self.start = torch.from_numpy(np.cumsum(n_series) - n_series).to(DEVICE)
            self.pix = torch.empty((len(self.rows), PATCH, PATCH), dtype=torch.int16, device=DEVICE)
            for s in range(0, len(self.rows), 1024):          # fewer than 2 ** 15 series per cluster (cluster_ndvi())
                self.pix[s:s + 1024] = torch.from_numpy(read_pixels(self.rows[s:s + 1024]).astype(np.int16)).to(DEVICE)
            gb_ndvi = (self.vals.numel() + self.pix.numel()) * 2 / 1e9
            where = 'in GPU memory' if DEVICE.type == 'cuda' else 'in memory'
            print(f'{len(INPUT)} bands of {len(self.rows)} clusters. Landsat and NL, {gb:.1f} GB: ' + (
                  where if self.x is not None else
                  f'more than GPU_DATA_GB = {GPU_DATA_GB}, so every batch reads them from the caches') +
                  f'; {len(NDVI_INPUT)} NDVI bands of {len(self.vals)} distinct series, {gb_ndvi:.1f} GB: {where}')

    def ndvi(self, idx):
        """The NDVI bands (float16 (len(idx), len(NDVI_INPUT), 224, 224)) of the clusters at positions idx (on
        DEVICE)."""
        series = self.pix[idx].long() + self.start[idx].view(-1, 1, 1)
        return self.vals[series].permute(0, 3, 1, 2)

    def images(self, idx):
        """Normalised float32 bands (channels last) of the clusters at positions idx (a 1-D tensor)."""
        if self.x is not None:
            x = self.x[idx.to(self.x.device)]
        else:
            x = torch.from_numpy(read_base(self.rows[idx.cpu().numpy()])).to(DEVICE, non_blocking=True)
        idx = idx.to(DEVICE)
        x = normalise(x.float(), self.nl_mean[idx], self.nl_std[idx])
        x = torch.cat([x, self.ndvi(idx).float()], dim=1)            # INPUT order: BASE, then NDVI_INPUT
        return x.contiguous(memory_format=torch.channels_last)

    def batch(self, idx, needs):
        """(bands or None, SatCLIP values or None) of the clusters at positions idx, for a model with inputs `needs`."""
        x = self.images(idx) if 'image' in needs else None
        loc = self.loc[idx.to(self.loc.device)] if 'location' in needs else None
        return x, loc


def r2_score(y, p):
    y, p = np.asarray(y, np.float64), np.asarray(p, np.float64)
    return float(1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=AMP and DEVICE.type == 'cuda')


NO_DECAY = ('relative_position', 'cpb_mlp', 'logit_scale', 'month_embed', 'channel_embed')   # Swin v2, OlmoEarth


def make_optimizer(model, recipe):
    """adam: Adam with LR for all weights and the Keras epsilon (05-07). adamw: AdamW as in 08, LR_PRETRAINED for the
    pretrained weights (weight decay on their matrices, not on biases, norms, embeddings and position biases) and
    LR_NEW without weight decay for the new ones (Regressor.is_new). Every group keeps its base_lr for the schedule."""
    params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    if recipe == 'adam':
        opt = torch.optim.Adam([{'params': [p for _, p in params], 'lr': LR, 'name': 'all'}], eps=1e-7)
    else:
        groups = {'decay': [], 'no_decay': [], 'new': []}
        for name, p in params:
            if model.is_new(name):
                groups['new'].append(p)
            elif p.ndim >= 2 and not name.endswith(('pos_embed', 'cls_token')) and not any(t in name for t in NO_DECAY):
                groups['decay'].append(p)
            else:
                groups['no_decay'].append(p)
        spec = [{'params': groups['decay'], 'lr': LR_PRETRAINED, 'weight_decay': WEIGHT_DECAY, 'name': 'pretrained'},
                {'params': groups['no_decay'], 'lr': LR_PRETRAINED, 'weight_decay': 0.0,
                 'name': 'pretrained_no_decay'},
                {'params': groups['new'], 'lr': LR_NEW, 'weight_decay': 0.0, 'name': 'new'}]
        opt = torch.optim.AdamW([g for g in spec if g['params']])
    for g in opt.param_groups:
        g['base_lr'] = g['lr']
    return opt


def lr_factor(t):
    """adamw: learning-rate factor at epoch t (fractional): linear warm-up over WARMUP_EPOCHS, cosine decay to 0."""
    if t < WARMUP_EPOCHS:
        return t / WARMUP_EPOCHS
    return 0.5 * (1 + math.cos(math.pi * min(1.0, (t - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1))))


def predict(model, data, idx, needs):
    """Predictions (bfloat16 autocast on the GPU) of the clusters at positions idx, in batches of 2 x BATCH_SIZE."""
    model.eval()
    out = []
    batch = 2 * BATCH_SIZE
    with torch.no_grad(), autocast():
        for s in range(0, len(idx), batch):
            out.append(model(*data.batch(idx[s:s + batch], needs)).float())
    return torch.cat(out)


def extract(model, data, idx, needs):
    """Features (n, n_features) and predictions (n,) of the clusters at positions idx, in float32 (no autocast), in
    batches of BATCH_SIZE (float32 needs more memory than training in bfloat16)."""
    model.eval()
    batch = BATCH_SIZE
    feats, preds = [], []
    with torch.no_grad():
        for s in range(0, len(idx), batch):
            f = model.features(*data.batch(idx[s:s + batch], needs)).float()
            feats.append(f.cpu())
            preds.append(model.head(f).squeeze(1).cpu())
    return torch.cat(feats).numpy(), torch.cat(preds).numpy()


def save(obj, path):
    torch.save(obj, path + '.tmp')
    os.replace(path + '.tmp', path)


def write_csv(frame, path, **kw):
    """Write a CSV file under a temporary name, then rename it: nobody (another run) reads half a file."""
    tmp = f'{path}.{uuid.uuid4().hex[:8]}.tmp'
    frame.to_csv(tmp, index=False, **kw)
    os.replace(tmp, path)


def check_start(from_scratch, where):
    """A run folder holds one kind of run: stop instead of mixing runs from pretrained and from random weights."""
    if bool(from_scratch) != TRAIN_FROM_SCRATCH:
        kind = {True: 'from random weights', False: 'from the pretrained weights'}
        raise RuntimeError(f'{where} holds a run {kind[bool(from_scratch)]}, but this run starts '
                           f'{kind[TRAIN_FROM_SCRATCH]}: give it another RUN_ROOT')

In [ ]:
META = ['cluster_id', 'country', 'year', 'rural', 'lat', 'lon', 'nl_sensor', 'fold']   # columns of predictions.csv


def split(y, folds, test_fold):
    """Positions of the training, validation and test clusters (with IWI) of a test fold, its validation fold and
    training folds."""
    k = FOLDS.index(test_fold)
    val_fold = FOLDS[(k - 1) % len(FOLDS)]
    train_folds = [f for f in FOLDS if f not in (test_fold, val_fold)]
    has_y = np.isfinite(y)
    return (np.flatnonzero(has_y & np.isin(folds, train_folds)), np.flatnonzero(has_y & (folds == val_fold)),
            np.flatnonzero(has_y & (folds == test_fold)), val_fold, train_folds)


def write_outputs(key, run_dir, data, y, folds, test_fold, pred, feats=None, extra=None):
    """predictions.csv (every cluster of data, all sets), features.csv (SAVE_FEATURES and feats), test_predictions.csv
    and result.json of a finished model of one test fold. Returns the result."""
    tr, va, te, val_fold, train_folds = split(y, folds, test_fold)
    role = np.select([np.isin(folds, train_folds), folds == val_fold, folds == test_fold], ['train', 'val', 'test'], '')
    n_set = {r: int((role == r).sum()) for r in ('train', 'val', 'test', '')}      # rows per set (incl. no IWI)
    out = pd.concat([clusters.iloc[data.rows][[c for c in META if c in clusters]].reset_index(drop=True),
                     pd.DataFrame({'role': role, TARGET: y, f'{TARGET}_pred': pred})], axis=1)
    write_csv(out.sort_values('cluster_id'), os.path.join(run_dir, 'predictions.csv'), float_format='%.7g')
    if feats is not None and SAVE_FEATURES:
        cols = pd.DataFrame(feats, columns=[f'pk_{key}_feat_{j}' for j in range(feats.shape[1])])
        write_csv(pd.concat([out, cols], axis=1).sort_values('cluster_id'), os.path.join(run_dir, 'features.csv'),
                  float_format='%.7g')
    p_te, y_te = pred[te], y[te]
    write_csv(clusters.iloc[data.rows[te]][['cluster_id', 'country', 'year', 'nl_sensor']].assign(y_true=y_te,
                                                                                                  y_pred=p_te),
              os.path.join(run_dir, 'test_predictions.csv'))
    name, _, recipe, needs = MODELS[key]
    result = {'key': key, 'model': name, 'start': start_of(key), 'recipe': recipe, 'inputs': '+'.join(needs),
              'test_fold': test_fold, 'val_fold': val_fold, 'train_folds': ','.join(train_folds),
              'keras_name': KERAS_NAME[test_fold], 'sensors': ','.join(NL_SENSORS), 'n_train': len(tr),
              'n_val': len(va), 'n_test': len(te), 'n_clusters': len(out),
              'n_features': None if feats is None else int(feats.shape[1]), 'val_r2': r2_score(y[va], pred[va]),
              'test_r2': r2_score(y_te, p_te), 'test_rmse': float(np.sqrt(((p_te - y_te) ** 2).mean())),
              'test_mae': float(np.abs(p_te - y_te).mean()), 'predictions_train': n_set['train'],
              'predictions_val': n_set['val'], 'predictions_test': n_set['test'], 'from_scratch': TRAIN_FROM_SCRATCH,
              'align': ALIGN, 'align_tag': ALIGN_TAG, 'peak_index': PEAK_INDEX, 'peak_window': PEAK_WINDOW,
              'input_set': INPUT_SET, 'n_bands': len(INPUT), **(extra or {})}
    result_file = os.path.join(run_dir, 'result.json')
    with open(result_file + '.tmp', 'w') as f:
        json.dump(result, f, indent=1)
    os.replace(result_file + '.tmp', result_file)
    print(f"{key} test {test_fold} ({KERAS_NAME[test_fold]}): test R2 {result['test_r2']:.3f}, RMSE "
          f"{result['test_rmse']:.2f}" + (f" (best epoch {result['best_epoch']})" if 'best_epoch' in result else '')
          + f"\n    predictions of all sets -> {run_dir}: train {n_set['train']} ({', '.join(train_folds)}), val "
          f"{n_set['val']} ({val_fold}), test {n_set['test']} ({test_fold})" + (f", no fold {n_set['']}"
                                                                                 if n_set[''] else ''))
    return result


def finished(key, f):
    """A model of one test fold is done: result.json, and features.csv for a network with SAVE_FEATURES."""
    run_dir = os.path.join(RUN_ROOT, key, f'test_{f}')
    need = ['result.json'] + (['features.csv'] if SAVE_FEATURES and MODELS[key][2] != 'ridge' else [])
    return all(os.path.exists(os.path.join(run_dir, n)) for n in need)


def run_fold(key, data, y, folds, test_fold):
    """Train (or continue) the network `key` of one test fold, then write the predictions (and features) of every
    cluster, the test predictions and result.json. y: IWI at the positions of data.rows (NaN: none); folds: their
    folds of 03 ('': none). Returns the test metrics (also in result.json)."""
    if MODELS[key][2] == 'ridge':
        return run_mosaiks(key, data, y, folds, test_fold)
    _, _, recipe, needs = MODELS[key]
    run_dir = os.path.join(RUN_ROOT, key, f'test_{test_fold}')
    result_file = os.path.join(run_dir, 'result.json')
    if os.path.exists(result_file):
        with open(result_file) as f:
            done = json.load(f)
        check_start(done.get('from_scratch', False), run_dir)
        if finished(key, test_fold):
            return done
    os.makedirs(run_dir, exist_ok=True)
    k = FOLDS.index(test_fold)
    tr, va, te, val_fold, train_folds = (torch.from_numpy(v).to(DEVICE) if isinstance(v, np.ndarray) else v
                                         for v in split(y, folds, test_fold))
    y_dev = torch.tensor(np.nan_to_num(y), dtype=torch.float32, device=DEVICE)
    assert len(tr) >= BATCH_SIZE, f'{key} test {test_fold}: {len(tr)} training clusters, fewer than BATCH_SIZE'

    torch.manual_seed(SEED + k)
    model = build_model(key).to(DEVICE)
    if 'image' in needs:
        model = model.to(memory_format=torch.channels_last)
    if 'location' in needs:
        model.set_location_scale(data.loc[tr])
    with torch.no_grad():
        model.head.bias.fill_(float(y_dev[tr].mean()))
    opt = make_optimizer(model, recipe)
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(SEED * 100 + k)
    start, best, best_epoch = 0, float('inf'), -1
    last_file, best_file, hist_file = (os.path.join(run_dir, f) for f in ('last.pt', 'best.pt', 'history.csv'))
    if os.path.exists(result_file):                     # finished before: only the features are written again
        assert os.path.exists(best_file), f'{best_file} is missing: delete {result_file} to train this model again'
        start = EPOCHS
    elif os.path.exists(last_file):                     # continue a run that was stopped
        st = torch.load(last_file, map_location=DEVICE)
        check_start(st.get('from_scratch', False), run_dir)
        model.load_state_dict(st['model'])
        opt.load_state_dict(st['opt'])
        gen.set_state(st['gen'].cpu())
        start, best, best_epoch = st['epoch'] + 1, st['best'], st['best_epoch']
        if os.path.exists(hist_file):                   # epochs after the saved state are trained again
            with open(hist_file) as fh:
                kept = fh.readlines()[:start + 1]       # the header and epochs 1 ... start, as they were written
            with open(hist_file, 'w') as fh:
                fh.writelines(kept)
        print(f'{key} test {test_fold}: continuing at epoch {start + 1}')
    elif os.path.exists(hist_file):
        os.remove(hist_file)                            # history of a run that stopped before saving its state
    if start == 0:
        print(f'{key} test {test_fold}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters, '
              f'{model.n_features} features, {recipe}, start: {start_of(key)}')

    n_steps = max(len(tr) // BATCH_SIZE, 1)
    for epoch in range(start, EPOCHS):
        t0 = time.time()
        model.train()
        perm = tr[torch.randperm(len(tr), generator=gen, device=DEVICE)]
        sq_sum = torch.zeros((), device=DEVICE)
        n_seen = 0
        for j, s in enumerate(range(0, len(perm) - BATCH_SIZE + 1, BATCH_SIZE)):    # last incomplete batch dropped
            if recipe == 'adamw':                       # warm-up and cosine decay, every step
                for g in opt.param_groups:
                    g['lr'] = g['base_lr'] * lr_factor(epoch + (j + 1) / n_steps)
            b = perm[s:s + BATCH_SIZE]
            x, loc = data.batch(b, needs)
            with autocast():
                pred = model(x, loc)
            mse = F.mse_loss(pred.float(), y_dev[b])
            loss = mse + L2_HEAD * model.head.weight.pow(2).sum() if L2_HEAD else mse
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sq_sum += mse.detach() * len(b)
            n_seen += len(b)
        reg = L2_HEAD * float(model.head.weight.detach().pow(2).sum())
        p_va = predict(model, data, va, needs).cpu().numpy()
        y_va = y[va.cpu().numpy()]
        val_mse = float(((p_va - y_va) ** 2).mean())
        val_loss = val_mse + reg
        if val_loss < best:
            best, best_epoch = val_loss, epoch
            save({'model': model.state_dict(), 'epoch': epoch, 'from_scratch': TRAIN_FROM_SCRATCH}, best_file)
        lrs = {f"lr_{g['name']}": g['lr'] for g in opt.param_groups}
        rec = {'epoch': epoch + 1, 'train_mse': float(sq_sum) / max(n_seen, 1), 'l2_term': reg, 'val_mse': val_mse,
               'val_loss': val_loss, 'val_r2': r2_score(y_va, p_va), **lrs, 'seconds': round(time.time() - t0, 1)}
        pd.DataFrame([rec]).to_csv(hist_file, mode='a', header=not os.path.exists(hist_file), index=False)
        save({'model': model.state_dict(), 'opt': opt.state_dict(), 'gen': gen.get_state(), 'epoch': epoch,
              'best': best, 'best_epoch': best_epoch, 'from_scratch': TRAIN_FROM_SCRATCH}, last_file)
        if epoch == start or (epoch + 1) % 10 == 0 or epoch + 1 == EPOCHS:
            print(f"{key} test {test_fold} epoch {epoch + 1}/{EPOCHS}: train mse {rec['train_mse']:.2f}, val mse "
                  f"{val_mse:.2f}, val R2 {rec['val_r2']:.3f}, best epoch {best_epoch + 1}, {rec['seconds']} s")

    # the best weights: features and prediction of every cluster (all sets), test metrics
    st = torch.load(best_file, map_location=DEVICE)
    check_start(st.get('from_scratch', False), run_dir)
    model.load_state_dict(st['model'])
    feats, pred = extract(model, data, torch.arange(len(data.rows), device=DEVICE), needs)
    result = write_outputs(key, run_dir, data, y, folds, test_fold, pred, feats,
                           {'best_epoch': st['epoch'] + 1, 'epochs': EPOCHS})
    if os.path.exists(last_file):
        os.remove(last_file)                            # only needed to continue a stopped run
    return result

In [ ]:
def band_scaling(data, idx, n_max=1000):
    """Mean and standard deviation of every input band over the pixels of (up to n_max of) the clusters at positions
    idx (a 1-D tensor), float32 tensors on DEVICE."""
    pick = idx[torch.randperm(len(idx), generator=torch.Generator().manual_seed(SEED))[:n_max].to(idx.device)]
    s1 = torch.zeros(len(INPUT), dtype=torch.float64, device=DEVICE)
    s2 = torch.zeros_like(s1)
    for s in range(0, len(pick), 64):
        x = data.images(pick[s:s + 64]).double()
        s1 += x.sum(dim=(0, 2, 3))
        s2 += (x ** 2).sum(dim=(0, 2, 3))
    n = len(pick) * PATCH * PATCH
    mean = s1 / n
    return mean.float(), (s2 / n - mean ** 2).clamp(min=1e-12).sqrt().float()


class ScaledImages:
    """The standardised bands of the clusters at positions idx of `data` as a dataset for torchgeo's RCF:
    item i = {'image': tensor (bands, 224, 224)}."""

    def __init__(self, data, idx, mean, std):
        self.data, self.idx = data, idx
        self.mean, self.std = mean.view(1, -1, 1, 1), std.view(1, -1, 1, 1)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        x = (self.data.images(self.idx[int(i):int(i) + 1]) - self.mean) / self.std
        return {'image': x[0].float().cpu()}


def ridge(X, y, alphas):
    """Ridge regression of y on X with intercept (not penalised) for every alpha, the features standardised with the
    mean and standard deviation of X (constant features: scale 1), as in 09; float64 tensors. Returns the coefficients
    on the raw features (len(alphas), n_features) and the intercepts (len(alphas),)."""
    mean, std = X.mean(dim=0), X.std(dim=0, unbiased=False)
    std = torch.where(std < 1e-12, torch.ones_like(std), std)
    Z = (X - mean) / std
    y_mean = y.mean()
    s, V = torch.linalg.eigh(Z.T @ Z)
    s = s.clamp(min=0)
    Vb = V.T @ (Z.T @ (y - y_mean))
    coef = torch.stack([V @ (Vb / (s + a)) for a in alphas]) / std
    return coef, y_mean - coef @ mean


def run_mosaiks(key, data, y, folds, test_fold):
    """MOSAIKS of one test fold: band scaling and RCF filters from the training clusters, the features of every
    cluster, ridge (alpha on the validation fold, then fitted on the training and validation folds), outputs."""
    run_dir = os.path.join(RUN_ROOT, key, f'test_{test_fold}')
    result_file = os.path.join(run_dir, 'result.json')
    if os.path.exists(result_file):
        with open(result_file) as f:
            return json.load(f)
    os.makedirs(run_dir, exist_ok=True)
    from torchgeo.models import RCF
    t0 = time.time()
    k = FOLDS.index(test_fold)
    tr, va, te, _, _ = split(y, folds, test_fold)
    idx_tr = torch.from_numpy(tr).to(DEVICE)
    mean, std = band_scaling(data, idx_tr)
    torch.manual_seed(SEED + k)
    rcf = RCF(in_channels=len(INPUT), features=MOSAIKS_FEATURES, kernel_size=MOSAIKS_KERNEL, bias=MOSAIKS_BIAS,
              seed=SEED + k + 1, mode='empirical', dataset=ScaledImages(data, idx_tr, mean, std)).to(DEVICE)
    X = torch.empty((len(data.rows), MOSAIKS_FEATURES), dtype=torch.float64, device=DEVICE)
    with torch.no_grad():
        for s in tqdm(range(0, len(data.rows), MOSAIKS_BATCH), desc=f'MOSAIKS features, test {test_fold}', leave=False):
            idx = torch.arange(s, min(s + MOSAIKS_BATCH, len(data.rows)), device=DEVICE)
            x = (data.images(idx) - mean.view(1, -1, 1, 1)) / std.view(1, -1, 1, 1)
            X[s:s + len(idx)] = rcf(x.contiguous()).double()
    t_feat = time.time() - t0
    yd = torch.tensor(np.nan_to_num(y), dtype=torch.float64, device=DEVICE)
    tr_d, va_d = torch.from_numpy(tr).to(DEVICE), torch.from_numpy(va).to(DEVICE)
    coef, icpt = ridge(X[tr_d], yd[tr_d], ALPHAS)                        # alpha: fitted on train, tested on val
    p_va = X[va_d] @ coef.T + icpt
    a = int(torch.argmin(((p_va - yd[va_d, None]) ** 2).mean(dim=0)))
    alpha, val_r2 = float(ALPHAS[a]), r2_score(y[va], p_va[:, a].cpu().numpy())
    fit = torch.from_numpy(np.concatenate([tr, va])).to(DEVICE)
    coef, icpt = ridge(X[fit], yd[fit], [alpha])                         # final: train + val
    pred = (X @ coef[0] + icpt[0]).cpu().numpy()
    save({'filters': rcf.weights.cpu(), 'biases': rcf.biases.cpu(), 'band_mean': mean.cpu(), 'band_std': std.cpu(),
          'alpha': alpha, 'coef': coef[0].cpu(), 'intercept': float(icpt[0])}, os.path.join(run_dir, 'mosaiks.pt'))
    edge = alpha in (ALPHAS[0], ALPHAS[-1])
    if edge:
        print(f'NOTE: {key} test {test_fold}: alpha {alpha:g} is at the edge of ALPHAS; widen ALPHAS')
    print(f'{key} test {test_fold}: {MOSAIKS_FEATURES} features of {len(data.rows)} clusters in {t_feat:.0f} s, '
          f'alpha {alpha:g} (validation R2 {val_r2:.3f})')
    return write_outputs(key, run_dir, data, y, folds, test_fold, pred, None,
                         {'n_features': MOSAIKS_FEATURES, 'alpha': alpha, 'alpha_at_grid_edge': edge,
                          'val_r2': val_r2, 'seconds': round(time.time() - t0, 1)})

# 5. Train

The models train on the clusters with complete data, an IWI value and a fold of 03, and predict all clusters with complete data. The input bands are loaded once for all models (the NDVI bands as the values of the distinct series and the pixel index). Models and test folds come from `RUN_MODELS` and `RUN_FOLDS`.

In [ ]:
rows = np.flatnonzero(complete)                                # rows of `clusters` that get predictions
y = clusters[TARGET].to_numpy(dtype=np.float64)[rows]          # NaN: no IWI, not trained on
folds = clusters['fold'].fillna('').astype(str).to_numpy()[rows]
trainable = np.isfinite(y) & np.isin(folds, FOLDS)
print(f'{len(rows)} clusters with complete data, {trainable.sum()} of them with {TARGET} and a fold:',
      pd.Series(folds[trainable]).value_counts().sort_index().to_dict())

runs = {key: [f for f in RUN_FOLDS if not finished(key, f)] for key in todo}
needs = {n for key, missing in runs.items() if missing for n in MODELS[key][3]}
data = Inputs(rows, load_images='image' in needs) if needs else None
if 'location' in needs:
    data.loc = satclip_embeddings(rows)
    print(f'SatCLIP values of {len(rows)} clusters: {tuple(data.loc.shape)}')
if needs and DEVICE.type != 'cuda':
    print('WARNING: no GPU, training will be very slow')
for key in todo:
    if not runs[key]:
        print(f'{key}: test folds {RUN_FOLDS} already done')
        continue
    print(f'\n=== {key}: {MODELS[key][0]}, start: {start_of(key)}, test folds {runs[key]}')
    for f in runs[key]:
        run_fold(key, data, y, folds, f)
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
if not todo:
    print('PK_MODELS=none: no training, only the results')

# 6. Results

For every model: the mean and standard deviation of the test R² over the test folds that are done, and the R², RMSE and MAE (in IWI points) of all test predictions pooled, also for each period. Next to them, for every run of `COMPARE` that is done: its mean test R² (`R2 <run>`) and the **gain** of this run over it, the test R² of this run minus that of the other for every test fold that both have done with the same training, validation and test clusters (`gain vs <run>`: mean, `sd` and number of `folds`). The runs are 11 (`11`: the 6 wavelet bands of the 12 calendar months), this notebook with the other alignments of the same input set (`calendar`: `PK_ALIGN = none`; `pixel11`, `cluster11`), 13 (`13`: the 5 DFT bands, blind to the date of the season) and 12 (`12`: no NDVI band); `gain vs calendar` is the alignment gain. For comparison, also the mean test R² of 06 (NL only), 07 (`landsat_wavelet6`: Landsat + wavelet bands) and of the ridge regression of 09 on their features, on the same clusters, when they are done (and of 05's `base_wavelet6` when `NL_SENSORS = ['VIIRS']`). The tables go to `pk_results.csv` and `pk_per_fold.csv` in `RUN_ROOT`, and `oof_predictions.csv` has every test cluster once, with the prediction of each model whose test fold it is in (columns = model keys).

The plot shows the test R² of every model (left) and its gain over the other runs (right; the first four that are done, in the order above): a gain whose bar of ± 1 sd over the folds crosses zero is within the spread of the folds.

Reading the predictions, or the features of a model, for example `resnet50` tested on fold E, and merging them with the features of the same model in 11:

```python
pred = pd.read_csv(os.path.join(RUN_ROOT, 'resnet50', 'test_E', 'predictions.csv'))       # all clusters, role
feat = pd.read_csv(os.path.join(RUN_ROOT, 'resnet50', 'test_E', 'features.csv'))
X = feat.filter(like='pk_resnet50_feat_').to_numpy()                                      # (clusters, 2048)
f11 = pd.read_csv(os.path.join(COMPARE['11'], 'resnet50', 'test_E', 'features.csv'))
both = feat.merge(f11[['cluster_id'] + [c for c in f11 if c.startswith('resnet50_feat_')]], on='cluster_id')
```

In [ ]:
def scores(p):
    err = p['y_pred'] - p['y_true']
    return {'clusters': len(p), 'R2': r2_score(p['y_true'], p['y_pred']) if len(p) > 1 else np.nan,
            'RMSE': float(np.sqrt((err ** 2).mean())), 'MAE': float(err.abs().mean())}


def fold_results(key, run_root=None, check=True):
    """result.json of every finished test fold of a model, and its pooled test predictions (or None)."""
    found, preds = [], []
    for f in FOLDS:
        run_dir = os.path.join(run_root or RUN_ROOT, key, f'test_{f}')
        if os.path.exists(os.path.join(run_dir, 'result.json')):
            with open(os.path.join(run_dir, 'result.json')) as fh:
                found.append(json.load(fh))
            if check:                                  # a run folder holds one kind of run
                check_start(found[-1].get('from_scratch', False), run_dir)
            preds.append(pd.read_csv(os.path.join(run_dir, 'test_predictions.csv')))
    return found, (pd.concat(preds, ignore_index=True) if preds else None)


def reference_results():
    """[(label, mean test R2)] of the comparable runs of 05-09 that are done."""
    refs = []
    for start in REF_STARTS:
        f = os.path.join(WORK_DIR, 'nl_resnet18_runs', f'{PERIODS}_{start}', 'nl_resnet18_results.csv')
        if os.path.exists(f):
            t = pd.read_csv(f)
            t = t[t['what'].astype(str).str.startswith('mean of')]
            if len(t):
                refs.append((f'NL only, ResNet-18 (06, {start})', float(t['R2'].iloc[0])))
                break
    for start in REF_STARTS:
        f = os.path.join(WORK_DIR, 'landsat_resnet18_runs', f'{PERIODS}_{start}', 'landsat_resnet18_results.csv')
        if os.path.exists(f):
            t = pd.read_csv(f).set_index('key')
            if 'landsat_wavelet6' in t.index and t.at['landsat_wavelet6', 'folds done'] > 0:
                refs.append((f'Landsat + wavelets, ResNet-18 (07, {start})',
                             float(t.at['landsat_wavelet6', 'test R2 mean'])))
                break
    if PERIODS == 'DMSP_VIIRS' and os.path.exists(RIDGE_RESULTS):
        t = pd.read_csv(RIDGE_RESULTS)
        t = t[(t['key'] == 'landsat_wavelet6') & (t['features'] == 'NL + Landsat')]
        if len(t) and t['folds done'].iloc[0] > 0:
            refs.append(('ridge on the features of 06 + 07 (09)', float(t['test R2 mean'].iloc[0])))
    if PERIODS == 'VIIRS':
        r = [x['test_r2'] for x in fold_results('base_wavelet6', T2_RESNET18, check=False)[0]]
        if r:
            refs.append((f'base_wavelet6, ResNet-18 (05, {len(r)} folds)', float(np.mean(r))))
    return refs


per_fold, rows_out, oof = [], [], []
for key, (name, _, recipe, needs) in MODELS.items():
    found, p = fold_results(key)
    per_fold += found
    r = pd.Series([x['test_r2'] for x in found], dtype=float)
    out = {'key': key, 'model': name, 'start': start_of(key), 'training': recipe, 'folds done': len(r),
           'test R2 mean': r.mean() if len(r) else np.nan, 'test R2 sd': r.std() if len(r) > 1 else np.nan,
           'test R2 min': r.min() if len(r) else np.nan, 'test R2 max': r.max() if len(r) else np.nan}
    if p is not None:
        sc = scores(p)
        out.update({'pooled R2': sc['R2'], 'pooled RMSE': sc['RMSE'], 'pooled MAE': sc['MAE']})
        for s, ps in p.groupby('nl_sensor'):
            out[f'pooled R2 {s}'] = scores(ps)['R2']
        oof.append(p.set_index('cluster_id')['y_pred'].rename(key))
    rows_out.append(out)
# the same models in the runs of COMPARE (11, 12 and the other alignments of this input set): their mean test R2, and
# the gain = the test R2 of this run minus theirs, paired by test fold (folds with the same numbers of training,
# validation and test clusters in both)
found_compare = []
for c, root in COMPARE.items():
    n_found = 0
    for out in rows_out:
        mine = {x['test_fold']: x for x in per_fold if x['key'] == out['key']}
        theirs = {x['test_fold']: x for x in fold_results(out['key'], root, check=False)[0]}
        n_found += bool(theirs)
        gain = [mine[f]['test_r2'] - theirs[f]['test_r2'] for f in mine if f in theirs and
                all(theirs[f][n] == mine[f][n] for n in ('n_train', 'n_val', 'n_test'))]
        out[f'R2 {c}'] = np.mean([x['test_r2'] for x in theirs.values()]) if theirs else np.nan
        out[f'gain vs {c}'] = np.mean(gain) if gain else np.nan
        out[f'gain vs {c} sd'] = np.std(gain, ddof=1) if len(gain) > 1 else np.nan
        out[f'gain vs {c} folds'] = len(gain)
    if n_found:
        found_compare.append(c)
CMP_COLS = [f'{p}{c}{q}' for c in COMPARE for p, q in (('R2 ', ''), ('gain vs ', ''), ('gain vs ', ' sd'),
                                                       ('gain vs ', ' folds'))]
RES_COLS = ['key', 'model', 'start', 'training', 'folds done', 'test R2 mean', 'test R2 sd', 'test R2 min',
            'test R2 max', 'pooled R2', 'pooled RMSE', 'pooled MAE'] + [f'pooled R2 {s}' for s in NL_SENSORS] + CMP_COLS
results = pd.DataFrame(rows_out, columns=RES_COLS)
FOLD_COLS = ['key', 'test_fold', 'val_fold', 'train_folds', 'keras_name', 'n_train', 'n_val', 'n_test', 'best_epoch',
             'alpha', 'val_r2', 'test_r2', 'test_rmse', 'test_mae']
per_fold = pd.DataFrame(per_fold, columns=FOLD_COLS + ['model', 'start', 'recipe', 'n_features', 'n_clusters',
                                                    'sensors', 'epochs', 'align_tag', 'input_set', 'n_bands'])
os.makedirs(RUN_ROOT, exist_ok=True)
write_csv(results, os.path.join(RUN_ROOT, 'pk_results.csv'))
write_csv(per_fold, os.path.join(RUN_ROOT, 'pk_per_fold.csv'))

OOF = None
if oof:
    base = clusters.iloc[rows][[c for c in ['cluster_id', 'country', 'year', 'rural', 'nl_sensor', 'fold', TARGET]
                                if c in clusters]].set_index('cluster_id')
    OOF = base.join(pd.concat(oof, axis=1), how='inner').reset_index()
    write_csv(OOF.sort_values('cluster_id'), os.path.join(RUN_ROOT, 'oof_predictions.csv'), float_format='%.6g')

REFS = reference_results()
print(f'Landsat + NL + NDVI bands {INPUT_SET} ({ALIGN_TAG}) -> {TARGET} ({" + ".join(NL_SENSORS)} periods, start: '
      f'{START}), spatial folds of 03; the same models in {found_compare or "no other run yet"}; comparison: '
      + ('; '.join(f'{label} R2 {v:.3f}' for label, v in REFS) or 'no runs of 05-09 found'))
if OOF is not None:
    print(f'oof_predictions.csv: {len(OOF)} test clusters, models: {[k for k in MODELS if k in OOF]}')
display(per_fold[FOLD_COLS].round(3))
display(results.drop(columns=CMP_COLS).round(3))
# the comparisons that are done: mean test R2 of the other run and the gain of this run over it
results[['key', 'folds done', 'test R2 mean'] + [f'{p}{c}{q}' for c in found_compare for p, q in
                                                 (('R2 ', ''), ('gain vs ', ''), ('gain vs ', ' sd'),
                                                  ('gain vs ', ' folds'))]].round(3)

In [ ]:
CMP_STYLE = {'11': ('s', '11: the wavelet bands of the 12 calendar months'),
             'calendar': ('^', 'the 23 composites in calendar order: the alignment gain'),
             'pixel': ('P', 'aligned on the own peak of every pixel'),
             'cluster': ('X', 'aligned on the peak of the patch'),
             '13': ('D', '13: the DFT bands, blind to the date of the season'),
             '12': ('v', '12: no NDVI band')}
CMP_COLORS = ['#6250d6', '#eda100', '#e87ba4', '#008300']   # the 4 gains shown, in the order of COMPARE


def cmp_style(c):
    """Marker and label of a run of COMPARE (the alignments by the start of their tag: pixel11 -> pixel)."""
    return next(v for k, v in CMP_STYLE.items() if c == k or (k in ('pixel', 'cluster') and c.startswith(k)))


done = results[results['folds done'] > 0].iloc[::-1]
if len(done):
    shown = [c for c in found_compare if done[f'gain vs {c}'].notna().any()][:len(CMP_COLORS)]
    fig, axes = plt.subplots(1, 2 if shown else 1, figsize=(16 if shown else 10, 0.5 * len(done) + 2.3), sharey=True,
                             constrained_layout=True, squeeze=False,
                             gridspec_kw={'width_ratios': [3, 2] if shown else [1]})
    ax = axes[0, 0]
    yy = np.arange(len(done))
    handles = [ax.barh(yy, done['test R2 mean'], xerr=done['test R2 sd'].fillna(0), height=0.6, color='#2a78d6',
                       error_kw={'ecolor': '#52514e', 'elinewidth': 1, 'capsize': 2},
                       label='mean ± sd of the test folds')]
    for s, marker, color in (('VIIRS', 'o', '#1baf7a'), ('DMSP', 'D', '#eb6834')):
        col = f'pooled R2 {s}'
        if col in done and done[col].notna().any():
            handles.append(ax.scatter(done[col], yy, marker=marker, s=36, color=color, edgecolors='white',
                                      linewidths=1.2, zorder=3, label=f'pooled test predictions, {s} period'))
    for (label, v), style in zip(REFS, ['--', ':', '-.', (0, (6, 2))]):
        handles.append(ax.axvline(v, color='#52514e', linewidth=1, linestyle=style, label=f'{label}: {v:.2f}'))
    for k, v, e in zip(yy, done['test R2 mean'], done['test R2 sd'].fillna(0)):
        if v > 0.15:                                       # value inside the bar, clear of the markers
            ax.text(0.01, k, f'{v:.2f}', va='center', ha='left', color='white', fontsize=8)
        else:                                              # small or negative: right of zero or of the error bar
            ax.text(min(max(v + e, 0), 0.85) + 0.015, k, f'{v:.2f}' + (' (bar cut at -1)' if v < -1 else ''),
                    va='center', ha='left', color='#52514e', fontsize=8,
                    bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 1, 'alpha': 0.8})
    ax.set_yticks(yy)
    ax.set_yticklabels([f'{k}  ({m})' for k, m in zip(done['key'], done['model'])], fontsize=8)
    marks = ['test R2 mean'] + [c for c in done if c.startswith('pooled R2 ')]
    x_lo = np.nanmin(done[marks].to_numpy())                                     # every marker on the axes
    ax.set_xlim(max(min(0, x_lo - 0.05), -1), 1)                                 # a model below -1 is cut at -1
    ax.set_xlabel('test R²')
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.1 - 0.6 / len(done)), ncol=2,
              frameon=False, fontsize=8)
    ax.set_title(f'Landsat + NL + NDVI {INPUT_SET} ({ALIGN_TAG}) → {TARGET.upper()}, {" + ".join(NL_SENSORS)} '
                 f'clusters, start: {START}, spatial 5-fold CV', loc='left', fontsize=11)
    if shown:                                          # the gain over the other runs, paired by test fold
        gx = axes[0, 1]
        gx.axvline(0, color='#52514e', linewidth=1)
        reach = np.nanmax((done[[f'gain vs {c}' for c in shown]].abs().to_numpy()
                           + done[[f'gain vs {c} sd' for c in shown]].fillna(0).to_numpy()))
        lim = min(1.15 * max(reach, 0.01), 1.0)                   # a gain beyond +-1 is drawn at the edge
        for off, color, c in zip(np.linspace(0.24, -0.24, len(shown)) if len(shown) > 1 else [0.0], CMP_COLORS, shown):
            marker, label = cmp_style(c)
            gx.errorbar(done[f'gain vs {c}'].clip(-lim, lim), yy + off, xerr=done[f'gain vs {c} sd'].fillna(0),
                        fmt=marker, color=color, ecolor=color, elinewidth=1, capsize=2, markersize=7,
                        markeredgecolor='white', markeredgewidth=1, label=f'vs {c} ({label})')
        gx.set_xlim(-lim, lim)
        gx.set_xlabel(f'gain in test R² ({ALIGN_TAG} minus the other run), mean ± sd of the paired folds'
                      + (' (axis cut at ±1)' if reach > 1 / 1.15 else ''))
        gx.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1 - 0.6 / len(done)), ncol=1, frameon=False,
                  fontsize=8)
        gx.set_title('the same model in the other runs', loc='left', fontsize=11)
    for a in axes.flat:
        a.grid(axis='x', color='#e1e0d9', linewidth=0.8)
        a.set_axisbelow(True)
        for spine in ('top', 'right'):
            a.spines[spine].set_visible(False)
    png = os.path.join(RUN_ROOT, 'pk_r2.png')
    fig.savefig(png + '.tmp.png', dpi=150)
    os.replace(png + '.tmp.png', png)
    plt.show()
else:
    print('no finished models yet')

In [ ]:
# predicted against observed IWI, every test cluster once (oof_predictions.csv)
keys = [k for k in MODELS if OOF is not None and k in OOF]
if keys:
    ncol = min(5, len(keys))
    nrow = math.ceil(len(keys) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.2 * nrow), constrained_layout=True, squeeze=False)
    lo, hi = -10.0, 110.0                                  # IWI is 0-100; predictions outside are drawn at the edge
    for a, key in zip(axes.flat, keys):
        m = OOF[key].notna().to_numpy()
        obs, pred = OOF.loc[m, TARGET].to_numpy(float), OOF.loc[m, key].to_numpy(float)
        a.hexbin(obs, pred.clip(lo, hi), gridsize=40, mincnt=1, bins='log', cmap='Blues', extent=(lo, hi, lo, hi))
        a.plot([lo, hi], [lo, hi], color='#52514e', linewidth=0.8)
        a.set_xlim(lo, hi)
        a.set_ylim(lo, hi)
        out = int(((pred < lo) | (pred > hi)).sum())
        a.set_title(f'{key}: R² {r2_score(obs, pred):.3f} ({m.sum()}' + (f', {out} off the axes)' if out else ')'),
                    fontsize=9)
        a.set_xlabel(f'observed {TARGET}', fontsize=8)
        a.set_ylabel(f'predicted {TARGET}', fontsize=8)
        a.tick_params(labelsize=7)
    for a in axes.flat[len(keys):]:
        a.axis('off')
    fig.suptitle(f'Test predictions of every model (each cluster from the model of its test fold), {INPUT_SET}, '
                 f'{ALIGN_TAG}', fontsize=10)
    png = os.path.join(RUN_ROOT, 'pk_scatter.png')
    fig.savefig(png + '.tmp.png', dpi=150)
    os.replace(png + '.tmp.png', png)
    plt.show()